In [1]:
# =====================================================================================
#  CYBERSECURITY AWARENESS & THREAT INTELLIGENCE DASHBOARD  -  single Google Colab cell
#  DEFENSIVE / EDUCATIONAL ONLY. Synthetic data only. Nothing here ever contacts an
#  indicator (IP / domain / URL), executes a file, or scans a system.
#  Paste this whole cell into Colab and press Run.  A dashboard link is printed at the end.
# =====================================================================================
import os, re, sys, json, math, time, random, sqlite3, ipaddress, hashlib, threading, subprocess, pathlib, queue
import urllib.parse
from datetime import datetime, timedelta, timezone
from collections import defaultdict

try:
    import flask, pandas  # noqa
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "flask", "pandas"])
import pandas as pd
from flask import Flask, request, jsonify, g, Response
from werkzeug.exceptions import HTTPException
from werkzeug.serving import make_server

# ------------------------------------------------------------------ 0. PROJECT SETUP
BASE = pathlib.Path("Cybersecurity-Threat-Intelligence-Dashboard")
for _d in ["backend/services", "frontend", "awareness", "data", "tests", "screenshots", "reports", "docs"]:
    (BASE / _d).mkdir(parents=True, exist_ok=True)
DB_PATH = str(BASE / "data" / "ti_dashboard.db")
PORT = 8050
SEED = 2026
NOW = datetime.now(timezone.utc).replace(tzinfo=None, microsecond=0)
ANALYST_KEY = os.environ.get("TI_ANALYST_KEY", "analyst-demo-key")   # role: analyst (read + write)
VIEWER_KEY = os.environ.get("TI_VIEWER_KEY", "viewer-demo-key")      # role: viewer  (read only)
ROLES = {ANALYST_KEY: "analyst", VIEWER_KEY: "viewer"}
TS = "%Y-%m-%d %H:%M:%S"
fmt = lambda d: d.strftime(TS)
parse = lambda s: datetime.strptime(s[:19], TS)

# ------------------------------------------------------------------ 1. REFERENCE DATA
CATEGORIES = ["PHISHING", "MALWARE", "RANSOMWARE", "CREDENTIAL THREATS", "WEB THREATS", "NETWORK THREATS",
              "VULNERABILITY EXPOSURE", "SOCIAL ENGINEERING", "DATA EXPOSURE", "ACCOUNT SECURITY"]
ITYPES = ["IP_ADDRESS", "DOMAIN", "URL", "FILE_HASH", "EMAIL_DOMAIN", "CVE_ID"]
SEVERITIES = ["INFORMATIONAL", "LOW", "MEDIUM", "HIGH", "CRITICAL"]
STATUSES = ["NEW", "UNDER_REVIEW", "MONITORING", "CLOSED", "FALSE_POSITIVE"]
ALERT_STATUSES = ["NEW", "INVESTIGATING", "MONITORING", "RESOLVED", "FALSE_POSITIVE"]
SEV_SCORE = {"INFORMATIONAL": 10, "LOW": 30, "MEDIUM": 55, "HIGH": 80, "CRITICAL": 100}
SOURCES = {"Internal SOC": "A", "Security Vendor": "B", "Research Report": "B",
           "Public Threat Feed": "C", "Community Submission": "C", "Unknown Source": "D"}
REL_LABEL = {"A": "Highly Reliable", "B": "Usually Reliable", "C": "Fairly Reliable", "D": "Reliability Unknown"}
REL_SCORE = {"A": 100, "B": 75, "C": 50, "D": 25}
REGIONS = ["North America", "Europe", "Asia-Pacific", "Latin America", "Middle East & Africa", ""]
DOC_NETS = ["192.0.2", "198.51.100", "203.0.113"]           # RFC 5737 documentation ranges
SAFE_TLDS = ["example.com", "example.org", "example.net", "invalid"]
WORDS = ["login-check", "secure-verify", "account-update", "mail-portal", "docs-share", "pay-review", "vpn-gateway",
         "cloud-sync", "invoice-center", "helpdesk-reset", "sso-auth", "file-transfer", "update-service",
         "wallet-confirm", "hr-benefits", "shipment-track", "token-refresh", "payroll-view"]
CAT_TYPES = {
    "PHISHING": ["DOMAIN", "URL", "EMAIL_DOMAIN", "IP_ADDRESS"], "MALWARE": ["FILE_HASH", "DOMAIN", "IP_ADDRESS", "URL"],
    "RANSOMWARE": ["FILE_HASH", "IP_ADDRESS", "DOMAIN"], "CREDENTIAL THREATS": ["IP_ADDRESS", "DOMAIN", "URL"],
    "WEB THREATS": ["URL", "DOMAIN", "IP_ADDRESS"], "NETWORK THREATS": ["IP_ADDRESS", "DOMAIN"],
    "VULNERABILITY EXPOSURE": ["CVE_ID", "IP_ADDRESS"], "SOCIAL ENGINEERING": ["EMAIL_DOMAIN", "DOMAIN", "URL"],
    "DATA EXPOSURE": ["URL", "DOMAIN", "IP_ADDRESS"], "ACCOUNT SECURITY": ["IP_ADDRESS", "EMAIL_DOMAIN", "DOMAIN"]}
NAMES = {
    "PHISHING": ["Credential Phishing Campaign", "Fake Invoice Lure Cluster", "Lookalike Login Portal"],
    "MALWARE": ["Generic Loader Activity", "Info-Stealer Family (demo)", "Trojanized Installer Sample"],
    "RANSOMWARE": ["Ransomware Precursor Activity", "Extortion Notice Infrastructure", "Encryptor Sample (demo)"],
    "CREDENTIAL THREATS": ["Password Spray Source", "Credential Stuffing Wave", "Session Token Theft Pattern"],
    "WEB THREATS": ["Drive-by Redirect Chain", "Malicious Ad Redirector", "Compromised Site Injection"],
    "NETWORK THREATS": ["Beaconing Pattern", "Port Sweep Source", "Suspicious Outbound Channel"],
    "VULNERABILITY EXPOSURE": ["Exposed Service Awareness", "Unpatched Gateway Advisory", "Edge Device CVE Watch"],
    "SOCIAL ENGINEERING": ["Executive Impersonation Attempt", "Helpdesk Pretext Call Cluster", "Vendor Payment Scam"],
    "DATA EXPOSURE": ["Public Bucket Exposure", "Paste-Site Leak Mention", "Misconfigured Share Link"],
    "ACCOUNT SECURITY": ["Impossible Travel Sign-in", "MFA Fatigue Pattern", "Dormant Account Reactivation"]}
# ATT&CK mapping (Enterprise matrix terminology). Applied ONLY when behaviour was actually observed.
ATTACK = {
    "PHISHING": {"*": ("Initial Access", "Phishing", "T1566")},
    "MALWARE": {"FILE_HASH": ("Execution", "User Execution", "T1204"), "*": ("Command and Control", "Ingress Tool Transfer", "T1105")},
    "RANSOMWARE": {"*": ("Impact", "Data Encrypted for Impact", "T1486")},
    "CREDENTIAL THREATS": {"IP_ADDRESS": ("Credential Access", "Brute Force", "T1110"), "*": ("Credential Access", "Steal Web Session Cookie", "T1539")},
    "WEB THREATS": {"*": ("Initial Access", "Drive-by Compromise", "T1189")},
    "NETWORK THREATS": {"IP_ADDRESS": ("Discovery", "Network Service Discovery", "T1046"), "*": ("Command and Control", "Application Layer Protocol", "T1071")},
    "VULNERABILITY EXPOSURE": {"*": ("Initial Access", "Exploit Public-Facing Application", "T1190")},
    "SOCIAL ENGINEERING": {"*": ("Reconnaissance", "Phishing for Information", "T1598")},
    "DATA EXPOSURE": {"*": ("Exfiltration", "Exfiltration Over Web Service", "T1567")},
    "ACCOUNT SECURITY": {"IP_ADDRESS": ("Initial Access", "Valid Accounts", "T1078"), "*": ("Persistence", "Account Manipulation", "T1098")}}
TACTIC_ORDER = ["Reconnaissance", "Initial Access", "Execution", "Persistence", "Credential Access", "Discovery",
                "Command and Control", "Exfiltration", "Impact"]
ACTIONS = {
    "PHISHING": ["Review related internal logs", "Check for authorized sightings", "Review email-security telemetry", "Increase phishing awareness", "Monitor for related indicators"],
    "MALWARE": ["Check endpoint-protection telemetry for the hash (lookup only)", "Review software-install approvals", "Do NOT execute or open any sample", "Share findings with the endpoint team"],
    "RANSOMWARE": ["Verify backups are recent and restorable", "Review patch status of exposed services", "Check privileged-account activity", "Confirm incident-response contacts are current"],
    "CREDENTIAL THREATS": ["Review authentication logs for the source", "Confirm MFA is enforced on exposed accounts", "Check for password-reset spikes", "Promote passkeys / password managers"],
    "WEB THREATS": ["Review web-proxy logs (read-only)", "Confirm browser and plugin updates", "Check DNS-filter policy coverage", "Remind users about safe browsing"],
    "NETWORK THREATS": ["Review firewall and DNS logs for matches", "Check asset ownership of the internal host", "Validate segmentation rules", "Monitor for repeat observations"],
    "VULNERABILITY EXPOSURE": ["Confirm whether the product exists in the asset inventory", "Check patch availability", "Apply compensating controls if no patch", "Prioritise by exposure and asset criticality"],
    "SOCIAL ENGINEERING": ["Warn likely target groups", "Remind staff of out-of-band verification", "Review finance/helpdesk call-back procedures", "Collect user reports"],
    "DATA EXPOSURE": ["Confirm whether the data is genuinely ours", "Review sharing permissions", "Notify the data owner", "Review privacy obligations with the DPO"],
    "ACCOUNT SECURITY": ["Review sign-in logs for the account(s)", "Confirm MFA registration and methods", "Review recent device registrations", "Prompt user to verify recent activity"]}
CAT_INFO = {  # (description, common indicators, impact, defensive controls, awareness recommendation)
    "PHISHING": ("Deceptive messages that try to steal credentials or push unsafe actions.", "Look-alike domains, urgent wording, unexpected links/QR codes", "Account compromise, fraud, malware delivery", "Email filtering, DMARC/SPF/DKIM, MFA, report button", "Hover before clicking; verify requests through a trusted channel"),
    "MALWARE": ("Software built to disrupt, spy or gain unauthorised access.", "File hashes, staging domains, odd installers", "Data loss, persistence, further compromise", "Endpoint protection, allow-listing, patching", "Install only approved software; never open unexpected attachments"),
    "RANSOMWARE": ("Malware that disables access to systems/data and demands payment.", "Known loader hashes, extortion-site domains", "Downtime, data theft, extortion", "Offline backups, patching, least privilege, segmentation, IR plan", "Report odd file behaviour at once; never pay or contact attackers on your own"),
    "CREDENTIAL THREATS": ("Attempts to obtain or reuse usernames, passwords or tokens.", "Many failed logins from one source, stuffing patterns", "Account takeover, lateral access", "MFA, rate limiting, breached-password checks", "Use unique passphrases and a password manager"),
    "WEB THREATS": ("Risky or compromised websites and redirect chains.", "Suspicious URLs, redirectors, injected scripts", "Drive-by downloads, credential theft", "DNS/web filtering, patched browsers, content security policy", "Check the address bar; do not ignore certificate warnings"),
    "NETWORK THREATS": ("Unusual network behaviour such as scanning or beaconing.", "Repeated probes, periodic outbound connections", "Reconnaissance, covert channels", "Firewalling, segmentation, flow monitoring", "Report unknown devices; keep Wi-Fi and router firmware current"),
    "VULNERABILITY EXPOSURE": ("Known weaknesses in software that may be reachable.", "CVE IDs, exposed service banners (as data)", "Unauthorised access if left unpatched", "Patch management, asset inventory, risk-based prioritisation", "Apply updates promptly; turn on automatic updates"),
    "SOCIAL ENGINEERING": ("Manipulating people into giving access or information.", "Impersonation, pretext calls, urgent payment requests", "Fraud, data disclosure", "Verification procedures, call-back policies, training", "Pause, verify through an official channel, then act"),
    "DATA EXPOSURE": ("Sensitive data reachable by people who should not see it.", "Public links, open storage, leak mentions", "Privacy breach, regulatory impact", "Access reviews, DLP, secure sharing defaults", "Share via approved tools; collect only the data you need"),
    "ACCOUNT SECURITY": ("Signals that an account may be misused.", "Impossible travel, MFA prompts, dormant reactivation", "Fraud, data access, persistence", "MFA, conditional access, sign-in monitoring", "Deny unexpected MFA prompts and report them")}

# ------------------------------------------------------------------ 2. IOC VALIDATION ENGINE
_DOMAIN_RE = re.compile(r"^(?=.{1,253}$)(?:[a-z0-9](?:[a-z0-9-]{0,61}[a-z0-9])?\.)+[a-z]{2,24}$")
_CVE_RE = re.compile(r"^CVE-\d{4}-\d{4,7}$")
_EMAIL_RE = re.compile(r"^[a-z0-9._%+-]{1,64}@((?:[a-z0-9-]+\.)+[a-z]{2,24})$")
TYPE_MAP = {"IPV4": "IP_ADDRESS", "IPV6": "IP_ADDRESS", "DOMAIN": "DOMAIN", "URL": "URL", "MD5": "FILE_HASH",
            "SHA1": "FILE_HASH", "SHA256": "FILE_HASH", "CVE_ID": "CVE_ID", "EMAIL": "EMAIL_DOMAIN"}

def sanitize_text(s, maxlen=1000):
    """Strip HTML tags + control characters (defence against stored XSS)."""
    s = re.sub(r"<[^>]*>", "", str(s or ""))
    s = re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", s)
    return s.strip()[:maxlen]

def validate_indicator(value):
    """SYNTAX check only: 'is this well-formed?'  NOT 'is this malicious?'  Never touches the network."""
    out = {"valid": False, "indicator_type": "UNKNOWN", "normalized_value": "", "lookup_value": "", "validation_notes": []}
    notes = out["validation_notes"]
    if not isinstance(value, str) or not value.strip():
        notes.append("Empty input."); return out
    v = value.strip()
    if len(v) > 2048:
        notes.append("Input too long (max 2048 characters)."); return out
    rf = v.replace("hxxp", "http").replace("hXXp", "http").replace("[.]", ".").replace("(.)", ".").replace("[:]", ":")
    if rf != v:
        notes.append("Defanged indicator re-fanged for analysis only. It is never visited."); v = rf
    low = v.lower()

    def done(t, norm, lookup=None):
        out.update(valid=True, indicator_type=t, normalized_value=norm, lookup_value=lookup or norm)
        notes.append("Syntactically valid. This does NOT mean the indicator is malicious.")
        return out
    if _CVE_RE.match(v.upper()):
        return done("CVE_ID", v.upper())
    if re.fullmatch(r"[0-9a-f]+", low) and len(low) in (32, 40, 64):
        return done({32: "MD5", 40: "SHA1", 64: "SHA256"}[len(low)], low)
    if "://" in v:
        p = urllib.parse.urlparse(v)
        host = (p.hostname or "").lower().rstrip(".")
        if p.scheme.lower() not in ("http", "https", "ftp"):
            notes.append("Unsupported URL scheme."); return out
        ok_host = _DOMAIN_RE.match(host)
        if not ok_host:
            try: ipaddress.ip_address(host); ok_host = True
            except ValueError: pass
        if not ok_host:
            notes.append("URL host is not a valid domain or IP."); return out
        norm = f"{p.scheme.lower()}://{host}" + (f":{p.port}" if p.port else "") + (p.path or "") + (f"?{p.query}" if p.query else "")
        if host.endswith((".invalid", "example.com", "example.org", "example.net")):
            notes.append("Host uses an IANA-reserved demo name - safe for training.")
        return done("URL", norm)
    m = _EMAIL_RE.match(low)
    if m:
        return done("EMAIL", low, m.group(1))
    try:
        ip = ipaddress.ip_address(v)
        if any(str(ip).startswith(n + ".") for n in DOC_NETS):
            notes.append("Address is in an RFC 5737 documentation range - safe for demos.")
        return done("IPV4" if ip.version == 4 else "IPV6", str(ip))
    except ValueError:
        pass
    d = low.rstrip(".")
    if _DOMAIN_RE.match(d):
        if d.endswith((".invalid", "example.com", "example.org", "example.net")):
            notes.append("Reserved demo domain - safe for training.")
        return done("DOMAIN", d)
    notes.append("Not a recognised indicator format (IPv4/IPv6, domain, URL, MD5/SHA1/SHA256, e-mail, CVE ID).")
    return out

# ------------------------------------------------------------------ 3. RISK / CONFIDENCE / SOURCE RELIABILITY
def classify_score(s):
    return "INFORMATIONAL" if s <= 20 else "LOW" if s <= 40 else "MEDIUM" if s <= 60 else "HIGH" if s <= 80 else "CRITICAL"

def recency_score(last_seen, now=None):
    now = now or NOW
    return max(0.0, 100 - (now - last_seen).days * 100 / 90)

def frequency_score(n):
    return min(100.0, math.log2(max(n, 0) + 1) * 15)

def calculate_threat_risk(severity, confidence, last_seen, observations, source_reliability, related_alerts=0, context=0, now=None):
    """Risk 0-100 = how CONCERNING it may be. High risk != confirmed compromise."""
    comp = {"severity": SEV_SCORE[severity] * 0.30, "confidence": confidence * 0.25,
            "recency": recency_score(last_seen, now) * 0.15, "frequency": frequency_score(observations) * 0.10,
            "source_reliability": REL_SCORE[source_reliability] * 0.10,
            "context": min(100, related_alerts * 20 + context) * 0.10}
    score = int(round(max(0, min(100, sum(comp.values())))))
    return {"score": score, "classification": classify_score(score), "components": {k: round(v, 1) for k, v in comp.items()}}

def calculate_confidence(source_reliability, corroborating_sources, observations, age_days, has_context):
    """Confidence 0-100 = how much the EVIDENCE supports the intel (not how bad it is)."""
    rel = REL_SCORE[source_reliability]
    corro = min(100, 25 + (corroborating_sources - 1) * 30)
    obs = min(100, math.log2(observations + 1) * 16)
    fresh = max(0, 100 - age_days * 100 / 120)
    ctx = 100 if has_context else 30
    return int(round(max(0, min(100, rel * .35 + corro * .25 + obs * .15 + fresh * .15 + ctx * .10))))

def classify_entity(conf, risk, status, source, behavior):
    """Distinguish OBSERVATION / INDICATOR / THREAT / INCIDENT (ALERT is tracked in its own table)."""
    if source == "Internal SOC" and risk >= 75 and conf >= 80 and status == "UNDER_REVIEW" and behavior:
        return "INCIDENT"
    if conf < 40 or status == "FALSE_POSITIVE": return "OBSERVATION"
    if risk >= 61 and conf >= 60: return "THREAT"
    return "INDICATOR"

def map_attack(category, itype, behavior_observed, confidence):
    """Return (tactic, technique, id) ONLY when behavioural evidence justifies it."""
    if not behavior_observed or confidence < 50: return None
    m = ATTACK.get(category, {})
    return m.get(itype) or m.get("*")

def calculate_vulnerability_priority(cvss, asset_criticality, exposure, exploitation, business_impact=3):
    """Context-aware priority (0-100). CVSS alone should not decide patch order."""
    exp = {"INTERNET_FACING": 20, "INTERNAL": 9, "ISOLATED_TEST": 1}[exposure]
    expl = {"EXPLOITED_IN_WILD_DEMO": 25, "POC_PUBLIC_DEMO": 12, "NONE_OBSERVED": 0}[exploitation]
    score = cvss / 10 * 30 + expl + exp + asset_criticality / 5 * 15 + business_impact / 5 * 10
    score = int(round(min(100, score)))
    label = "P1 - Immediate" if score >= 80 else "P2 - This week" if score >= 60 else "P3 - Planned" if score >= 40 else "P4 - Routine"
    return {"score": score, "label": label}

# ------------------------------------------------------------------ 4. SYNTHETIC DATA GENERATORS
def sev_from_cvss(c): return "CRITICAL" if c >= 9 else "HIGH" if c >= 7 else "MEDIUM" if c >= 4 else "LOW"

def generate_vulnerabilities(rng, n=120):
    cats = ["Web Server", "VPN Appliance", "Email Gateway", "Operating System", "Database", "Browser", "Office Suite",
            "Identity Provider", "Container Runtime", "Router Firmware", "Backup Software", "Mail Client"]
    rows = []
    for i in range(n):
        cvss = round(min(10, max(2.0, rng.gauss(6.9, 1.8))), 1)
        expo = rng.choices(["INTERNET_FACING", "INTERNAL", "ISOLATED_TEST"], [35, 45, 20])[0]
        expl = rng.choices(["NONE_OBSERVED", "POC_PUBLIC_DEMO", "EXPLOITED_IN_WILD_DEMO"], [60, 28, 12])[0]
        crit, biz = rng.randint(1, 5), rng.randint(1, 5)
        pr = calculate_vulnerability_priority(cvss, crit, expo, expl, biz)
        cat = rng.choice(cats)
        rows.append({"cve_id": f"CVE-2099-{1001 + i}", "product_category": cat, "severity": sev_from_cvss(cvss), "cvss_score": cvss,
                     "published_date": (NOW - timedelta(days=rng.randint(1, 240))).strftime("%Y-%m-%d"),
                     "patch_available": rng.random() < .72, "exploitation_status_demo": expl,
                     "description": f"SYNTHETIC / DEMO ONLY - fictional weakness in a {cat.lower()} component. No technical or exploit details.",
                     "asset_criticality": crit, "exposure": expo, "business_impact": biz,
                     "priority_score": pr["score"], "priority_label": pr["label"]})
    return pd.DataFrame(rows)

def generate_threats(rng, vulns, n=2000):
    cve_ids = vulns["cve_id"].tolist()
    campaigns = [{"id": f"CMP-{k:04d}", "cat": rng.choice(CATEGORIES), "start": NOW - timedelta(days=rng.randint(3, 170))} for k in range(2, 142)]
    src_names, src_w = list(SOURCES), [15, 22, 12, 25, 16, 10]
    recs = []
    for i in range(1, n + 1):
        camp = rng.choice(campaigns) if (i > 3 and rng.random() < .45) else None
        cat = camp["cat"] if camp else rng.choice(CATEGORIES)
        it = rng.choice(CAT_TYPES[cat])
        w = rng.choice(WORDS); tld = rng.choice(SAFE_TLDS); num = rng.randint(1, 999)
        if it == "IP_ADDRESS": val = f"{rng.choice(DOC_NETS)}.{rng.randint(1, 254)}"
        elif it == "DOMAIN": val = f"{w}{num}.{tld}"
        elif it == "URL": val = f"https://{w}{num}.{tld}/{rng.choice(['verify', 'signin', 'download/update', 'portal/auth', 'files/view'])}"
        elif it == "FILE_HASH": val = hashlib.sha256(f"synthetic-{SEED}-{i}-{rng.random()}".encode()).hexdigest()
        elif it == "EMAIL_DOMAIN": val = f"mail-{w}{num}.{tld}"
        else: val = rng.choice(cve_ids)
        if camp:
            fs = camp["start"] + timedelta(days=rng.randint(0, 5), hours=rng.randint(0, 23))
        else:
            fs = NOW - timedelta(days=int(rng.betavariate(1.1, 2.4) * 175) + 1, hours=rng.randint(0, 23))
        fs = min(fs, NOW - timedelta(hours=3))
        ls = min(NOW - timedelta(hours=1), fs + timedelta(days=rng.randint(0, 35), hours=rng.randint(0, 23)))
        src = rng.choices(src_names, src_w)[0]
        recs.append({"threat_id": f"THR-2026-{i:03d}", "timestamp": fmt(fs + timedelta(minutes=rng.randint(1, 90))),
                     "threat_name": f"Synthetic {rng.choice(NAMES[cat])}", "threat_category": cat, "indicator_type": it,
                     "indicator_value": val, "source_name": src, "source_reliability": SOURCES[src],
                     "first_seen": fmt(fs), "last_seen": fmt(ls), "campaign_id": camp["id"] if camp else "",
                     "observation_count": min(400, max(1, int(rng.lognormvariate(1.8, 1.1)))),
                     "corroborating_sources": rng.choices([1, 2, 3, 4], [40, 30, 20, 10])[0],
                     "behavior_observed": rng.random() < .55, "has_context": rng.random() < .6,
                     "base_severity": rng.choices(SEVERITIES, [16, 26, 31, 20, 7])[0],
                     "country_or_region_optional": rng.choice(REGIONS),
                     "cve_id_optional": val if it == "CVE_ID" else (rng.choice(cve_ids) if cat == "VULNERABILITY EXPOSURE" else "")})
    # scenario records (the safe demo story from the project brief)
    scen = {0: ("Synthetic Credential Phishing Campaign", "PHISHING", "DOMAIN", "login-check.invalid"),
            1: ("Synthetic Credential Phishing Campaign - Infrastructure", "PHISHING", "IP_ADDRESS", "198.51.100.25"),
            2: ("Synthetic Credential Phishing Campaign - Lure URL", "PHISHING", "URL", "https://login-check.invalid/verify/account")}
    for k, (nm, c, t, v) in scen.items():
        recs[k].update(threat_name=nm, threat_category=c, indicator_type=t, indicator_value=v, source_name="Security Vendor",
                       source_reliability="B", campaign_id="CMP-0001", observation_count=37 - k * 9, behavior_observed=True,
                       first_seen=fmt(NOW - timedelta(days=34 - k)), last_seen=fmt(NOW - timedelta(days=2)),
                       timestamp=fmt(NOW - timedelta(days=34 - k)), cve_id_optional="")
    size = defaultdict(int)
    for r in recs:
        if r["campaign_id"]: size[r["campaign_id"]] += 1
    for r in recs:
        fs, ls = parse(r["first_seen"]), parse(r["last_seen"])
        cs = size.get(r["campaign_id"], 1)
        r["confidence_score"] = calculate_confidence(r["source_reliability"], r["corroborating_sources"], r["observation_count"], (NOW - ls).days, r["has_context"])
        ctx = (30 if r["behavior_observed"] else 0) + (15 if r["cve_id_optional"] else 0)
        risk = calculate_threat_risk(r["base_severity"], r["confidence_score"], ls, r["observation_count"], r["source_reliability"], max(0, cs - 1), ctx)
        r["risk_score"] = risk["score"]
        age = (NOW - ls).days
        if age > 60: r["status"] = rng.choices(STATUSES, [0, 5, 30, 55, 10])[0]
        elif r["confidence_score"] < 45: r["status"] = rng.choices(STATUSES, [35, 40, 5, 5, 15])[0]
        else: r["status"] = rng.choices(STATUSES, [30, 25, 30, 10, 5])[0]
    for k in range(3):
        recs[k].update(risk_score=78, confidence_score=85, status="MONITORING")
    out = []
    for r in recs:
        r["severity"] = classify_score(r["risk_score"])
        mp = map_attack(r["threat_category"], r["indicator_type"], r["behavior_observed"], r["confidence_score"])
        r["mitre_tactic_optional"], r["mitre_technique_optional"], r["mitre_technique_id_optional"] = mp if mp else ("", "", "")
        r["classification"] = classify_entity(r["confidence_score"], r["risk_score"], r["status"], r["source_name"], r["behavior_observed"])
        tag = " [SYNTHETIC / DEMO ONLY]"
        r["description"] = (f"Fictional {r['threat_category'].lower()} record for training. {r['indicator_type'].replace('_', ' ').title()} "
                            f"seen {r['observation_count']} time(s) across demo feeds." + tag)
        out.append(r)
    df = pd.DataFrame(out)
    cols = ["threat_id", "timestamp", "threat_name", "threat_category", "indicator_type", "indicator_value", "source_name", "confidence_score",
            "severity", "risk_score", "status", "first_seen", "last_seen", "country_or_region_optional", "description",
            "mitre_tactic_optional", "mitre_technique_optional", "cve_id_optional", "mitre_technique_id_optional", "campaign_id",
            "observation_count", "source_reliability", "classification", "behavior_observed", "corroborating_sources"]
    return df[cols]

# ------------------------------------------------------------------ 5. ALERT ENGINE + ALERT CORRELATION
def generate_threat_alert(t, cluster_size=1, vuln_priority=None):
    """Return ONE raw alert event (dict) when a rule fires, else None. Rules: risk, confidence, vuln priority, repeats, clusters."""
    if t["status"] in ("FALSE_POSITIVE",): return None
    rule = None
    if vuln_priority is not None and vuln_priority >= 70: rule = ("VULNERABILITY_PRIORITY", f"Linked CVE has high contextual patch priority ({vuln_priority}/100).")
    elif t["risk_score"] >= 65 and t["confidence_score"] >= 50: rule = ("HIGH_RISK_HIGH_CONFIDENCE", f"Risk {t['risk_score']} with confidence {t['confidence_score']} crossed alert thresholds.")
    elif t["observation_count"] >= 25 and t["risk_score"] >= 45: rule = ("REPEATED_OBSERVATION", f"Indicator observed {t['observation_count']} times.")
    elif cluster_size >= 3 and t["risk_score"] >= 55: rule = ("CORRELATED_INDICATORS", f"Indicator belongs to a cluster of {cluster_size} related indicators.")
    if not rule: return None
    return {"threat_id": t["threat_id"], "alert_type": rule[0], "severity": t["severity"], "risk_score": t["risk_score"],
            "confidence_score": t["confidence_score"], "description": rule[1] + " Indicator match is evidence for triage, not proof of compromise.",
            "status": {"NEW": "NEW", "UNDER_REVIEW": "INVESTIGATING", "MONITORING": "MONITORING", "CLOSED": "RESOLVED"}.get(t["status"], "NEW")}

def correlate_alerts(events, window_minutes=5):
    """Merge bursts of identical raw events (same threat + rule within the window) into ONE alert with observation_count."""
    merged, cur = [], {}
    for e in sorted(events, key=lambda x: (x["threat_id"], x["alert_type"], x["timestamp"])):
        key = (e["threat_id"], e["alert_type"])
        ts = parse(e["timestamp"])
        c = cur.get(key)
        if c and ts - c["_last"] <= timedelta(minutes=window_minutes):
            c["observation_count"] += 1; c["_last"] = ts
        else:
            c = dict(e, observation_count=1, _last=ts); cur[key] = c; merged.append(c)
    for c in merged: c.pop("_last", None)
    merged.sort(key=lambda x: x["timestamp"])
    return merged

def build_alerts(rng, threats_df, vulns_df):
    vp = dict(zip(vulns_df["cve_id"], vulns_df["priority_score"]))
    size = threats_df[threats_df.campaign_id != ""].groupby("campaign_id").size().to_dict()
    raw = []
    for t in threats_df.to_dict("records"):
        a = generate_threat_alert(t, size.get(t["campaign_id"], 1), vp.get(t["cve_id_optional"]) if t["cve_id_optional"] else None)
        if not a: continue
        bursts = 2 if rng.random() < .2 else 1
        per = max(1, min(60, t["observation_count"]) // bursts)
        for _ in range(bursts):
            start = parse(t["last_seen"]) - timedelta(hours=rng.randint(0, 72), minutes=rng.randint(0, 59))
            for _e in range(per):
                raw.append(dict(a, timestamp=fmt(start + timedelta(seconds=rng.randint(0, 270)))))
    alerts = correlate_alerts(raw)
    for i, a in enumerate(alerts, 1):
        a["alert_id"] = f"ALR-{i:05d}"
    return alerts, len(raw)

# ------------------------------------------------------------------ 6. AWARENESS CONTENT (modules, quiz, glossary, IR)
def _m(mid, title, cat, what, why, warn, safe, ifhappens): return {"module_id": mid, "title": title, "category": cat, "what": what, "why": why, "warning_signs": warn, "safe_practices": safe, "if_something_happens": ifhappens}
MODULES = [
    _m("M01", "Phishing awareness", "Phishing", "Phishing is a deceptive message that tries to make you click, log in, pay or share data.", "Most account compromises begin with a convincing message, not clever hacking.",
       ["Unexpected sender or look-alike address", "Urgent or threatening wording", "Request to enter a password or code", "Unexpected attachment, link or QR code", "Payment or gift-card request", "Impersonation of a boss or vendor"],
       ["Type the website address yourself or use a bookmark", "Check the real sender domain, not just the display name", "Verify unusual requests through a second, trusted channel", "Use the report-phishing button"], ["Do not click further", "Report it to IT/security", "If you entered a password, change it and tell IT immediately"]),
    _m("M02", "Password security", "Passwords", "Passwords are the first lock on most accounts. Long, unique passphrases make guessing and reuse attacks far harder.", "When one site is breached, attackers try the same password elsewhere.",
       ["Same password on many sites", "Short, guessable patterns", "Passwords stored in plain notes"], ["Use a password manager", "Use long passphrases (4+ random words)", "Never share passwords", "Change a password if a breach is reported"], ["Change the password and any site that reused it", "Turn on MFA", "Report suspected exposure"]),
    _m("M03", "Multi-factor authentication", "MFA", "MFA asks for a second proof of identity (app, key or passkey) in addition to a password.", "A strong password plus MFA is generally much safer than a password alone.",
       ["Unexpected MFA prompts", "Callers asking for one-time codes", "Codes sent that you did not request"], ["Prefer authenticator apps, security keys or passkeys (WebAuthn) over SMS", "Never read codes out to anyone", "Keep backup codes safely"], ["Deny the prompt", "Change your password", "Report to IT"]),
    _m("M04", "Social engineering", "Social Engineering", "Manipulating people - by phone, message or in person - into giving access or information.", "It targets trust and urgency, which technical controls cannot fully block.",
       ["Pressure to skip normal process", "Callers claiming to be IT or executives", "Requests for codes or secrets", "Strangers following you through secure doors"], ["Verify identity via official contact details", "Follow standard procedures even when pressured", "Never share OTP codes"], ["End the conversation", "Report it with the details you remember"]),
    _m("M05", "Safe browsing", "Safe Browsing", "Using the web while checking sites, downloads and prompts before trusting them.", "Look-alike sites and unsafe downloads are common entry points.",
       ["Misspelled addresses", "Certificate warnings", "Pop-ups demanding urgent action", "Downloads from unknown sites"], ["Check the address bar before logging in", "Download only from official sources", "Keep your browser updated"], ["Close the tab", "Run your approved scan", "Report if you entered data"]),
    _m("M06", "Secure Wi-Fi", "Wi-Fi", "Wireless networks can be shared, copied or poorly configured.", "Unknown hotspots can expose traffic or imitate trusted names.",
       ["Network names that duplicate a trusted one", "Open networks asking for logins", "Router still using default admin password"], ["Confirm the network name with staff", "Use a VPN or mobile hotspot for sensitive work", "Change router defaults and update firmware"], ["Disconnect", "Change passwords used while connected", "Tell IT"]),
    _m("M07", "Software updates", "Updates", "Updates fix known weaknesses in operating systems, browsers and apps.", "Attackers rely on people delaying patches for known flaws.",
       ["Update reminders ignored for weeks", "End-of-life software"], ["Turn on automatic updates", "Restart when asked", "Remove software you no longer use"], ["Update now", "Ask IT about unsupported software"]),
    _m("M08", "Ransomware awareness", "Ransomware", "Ransomware blocks access to files or systems and demands payment, often with data-theft threats.", "It causes downtime and data exposure, and prevention is far cheaper than recovery.",
       ["Unexpected macro-enabled attachments", "Files suddenly renamed or unreadable", "Ransom notes on screen"], ["Keep tested offline backups", "Patch promptly", "Use least privilege", "Enable MFA", "Segment networks", "Know your incident-response plan"], ["Disconnect from the network", "Do not engage with attackers", "Report to IT/security immediately"]),
    _m("M09", "USB and removable media", "Removable Media", "USB drives and other media can carry unwanted software.", "Found or gifted drives are a classic way to bypass email defences.",
       ["Unlabelled drives", "Drives left in public spaces", "Prompts to run files from a drive"], ["Use only approved, company-issued drives", "Never plug in found devices", "Encrypt drives holding sensitive data"], ["Do not open it; hand it to IT", "Report where it was found"]),
    _m("M10", "Data privacy", "Privacy", "Protecting personal and confidential information from misuse or exposure.", "Over-sharing leads to leaks, fraud and compliance problems.",
       ["Requests for more data than needed", "Public share links on sensitive files"], ["Share the minimum necessary", "Use approved tools with access controls", "Classify and label sensitive files"], ["Remove access", "Tell the data owner / privacy contact"]),
    _m("M11", "Mobile security", "Mobile", "Phones and tablets hold email, files and authentication apps.", "A lost or compromised phone can expose accounts and MFA.",
       ["Apps asking for unneeded permissions", "Texts with urgent links", "Unexpected device prompts"], ["Use screen lock and encryption", "Install apps from official stores only", "Keep the OS updated"], ["Report a lost device quickly so it can be locked", "Change passwords if compromised"]),
    _m("M12", "Remote work security", "Remote Work", "Working from home or travelling extends the office into less controlled places.", "Home networks and shared spaces carry more risk.",
       ["Shared devices for work", "Unsecured home router", "Screens visible in public"], ["Use work-managed devices and VPN", "Lock your screen", "Separate work and personal accounts"], ["Report lost devices or suspicious network behaviour"]),
    _m("M13", "Cloud account security", "Cloud", "Cloud services are protected mostly by identity: logins, tokens and sharing settings.", "A compromised cloud login can expose everything the account can reach.",
       ["Unknown devices or locations in sign-in history", "Unexpected sharing links", "Unrecognised app consents"], ["Use MFA", "Review sharing and connected apps regularly", "Sign out on shared devices"], ["Sign out all sessions", "Reset password", "Report to IT"]),
    _m("M14", "Incident reporting", "Incident Reporting", "Telling the right team quickly when something looks wrong.", "Early reports limit impact and protect colleagues.",
       ["Clicking or opening something suspicious", "Lost devices", "Unexpected prompts or account changes"], ["Know your reporting channel", "Report even if you are unsure", "Share facts: what, when, where"], ["Report immediately", "Keep the message/screenshot", "Follow IT's instructions"]),
    _m("M15", "AI-enabled scam awareness", "AI Scams", "Scammers can use AI to write polished messages, clone voices or create convincing fake videos.", "Old tells such as spelling mistakes are less reliable now.",
       ["Voice or video request for money or secrets", "Perfect-looking messages with unusual urgency", "Pressure to keep it secret"], ["Use a pre-agreed code word or call back on a known number", "Verify financial requests through a second channel", "Slow down - urgency is the tell"], ["Stop and verify", "Report the attempt"])]
_Q = [  # (category, question, [options], correct_index, explanation)
    ("Phishing", "You receive an unexpected message asking you to urgently verify your password. What should you do?", ["Click the link immediately", "Reply with your password", "Verify through an official, trusted channel", "Forward your credentials to a colleague"], 2, "Never act on urgent credential requests inside the message. Contact the service through a channel you already trust."),
    ("Phishing", "Which is a common phishing red flag?", ["A routine message in an existing thread", "A calendar invite from a known colleague", "Message from your ticket system you requested", "An urgent threat to close your account unless you click a link"], 3, "Urgency plus a link and a threat are classic pressure tactics."),
    ("Phishing", "An email contains an unexpected QR code to 're-activate your MFA'. Best response?", ["Scan it quickly", "Don't scan it; report it and verify through the official portal", "Scan on a personal phone", "Print it for colleagues"], 1, "QR codes hide the destination. Treat them like unknown links."),
    ("Passwords", "What is the best password approach?", ["One strong password everywhere", "Unique passphrases stored in a password manager", "A sticky note on the monitor", "Same password with one character changed"], 1, "Uniqueness limits damage from a breach; a manager makes it practical."),
    ("Passwords", "Why is reusing passwords risky?", ["It slows your computer", "It breaks MFA", "A breach on one site can unlock your other accounts", "It makes passwords shorter"], 2, "Attackers try leaked passwords on other services (credential stuffing)."),
    ("Passwords", "Which password is strongest?", ["Summer2024!", "P@ssw0rd1", "YourName1990", "A unique 5-random-word passphrase"], 3, "Length and randomness beat predictable substitutions."),
    ("MFA", "What does MFA add to a password?", ["A second, independent proof of identity", "A longer password", "A faster login", "A backup of your files"], 0, "MFA means a stolen password alone is not enough."),
    ("MFA", "You get an MFA approval prompt you did not trigger. What do you do?", ["Approve to stop the prompts", "Ignore it forever", "Deny it, change your password and report it", "Share the code with the caller"], 2, "Unexpected prompts can mean your password is known. Deny and report."),
    ("MFA", "Which is generally the most phishing-resistant sign-in method?", ["SMS codes", "Passkeys / security keys (WebAuthn)", "Email codes", "Security questions"], 1, "Passkeys are bound to the real site, so look-alike pages cannot capture them."),
    ("Social Engineering", "A caller says they are IT support and asks for your one-time code. What should you do?", ["Read the code out", "Hang up and contact IT via an official number", "Ask them to email you", "Give half the code"], 1, "IT will never need your one-time code. Verify independently."),
    ("Social Engineering", "Pretexting is...", ["A type of firewall", "A backup method", "Inventing a believable story to obtain information or access", "A password policy"], 2, "The attacker builds a false scenario to look legitimate."),
    ("Social Engineering", "A stranger follows you through a secure door saying they forgot their badge. Best action?", ["Hold the door", "Politely direct them to reception / security", "Give them your badge", "Ignore it"], 1, "Tailgating bypasses physical controls. Use the official visitor process."),
    ("Safe Browsing", "Before typing credentials, what should you check?", ["That the address is the expected domain you typed or bookmarked", "That the page has a padlock only", "That the logo looks right", "That the page loads fast"], 0, "Look-alike pages can have padlocks and perfect logos; the domain is what matters."),
    ("Safe Browsing", "Your browser shows a certificate warning on a login page. You should...", ["Click 'proceed anyway'", "Leave the page and report it if unexpected", "Disable the warning", "Try a different password"], 1, "Warnings may indicate impersonation or interception."),
    ("Safe Browsing", "Where should you download software from?", ["Links in chat messages", "Official vendor sites or approved app stores", "Any top search result", "File-sharing pages"], 1, "Official sources reduce the risk of tampered installers."),
    ("Ransomware", "What is the best protection against data loss from ransomware?", ["Pay quickly", "Regular, tested, offline or immutable backups", "Rename important files", "Disable antivirus"], 1, "Reliable backups let you restore without negotiating."),
    ("Ransomware", "A ransom note appears on your PC. What first?", ["Pay the ransom", "Disconnect from the network, don't contact attackers, and report to IT", "Delete all files", "Keep working"], 1, "Isolating the device limits spread; IT will coordinate recovery."),
    ("Ransomware", "Which attachment is highest risk?", ["A PDF from a colleague you were expecting", "An unexpected macro-enabled 'invoice' from an unknown sender", "A calendar file from your manager", "A plain text note"], 1, "Unexpected macro documents are a frequent delivery method."),
    ("Privacy", "Data minimisation means...", ["Collecting as much data as possible", "Collecting and sharing only what is needed", "Deleting all data weekly", "Encrypting passwords only"], 1, "Less data held means less data exposed."),
    ("Privacy", "How should you share a sensitive document with a colleague?", ["Personal email", "Approved tools with access controls", "Public link", "Chat with an unknown group"], 1, "Approved tools provide permissions and audit trails."),
    ("Privacy", "Why avoid posting photos of your work badge?", ["It uses storage", "It can help impersonation or cloning", "It reduces followers", "It is illegal everywhere"], 1, "Badge details can be used to forge access or impersonate you."),
    ("Wi-Fi", "On public Wi-Fi you need to do sensitive work. What is best?", ["Use any open network", "Use a trusted VPN or your phone hotspot", "Turn off the screen lock", "Share the password with friends"], 1, "A VPN or hotspot reduces exposure to rogue or monitored networks."),
    ("Wi-Fi", "You see two networks: 'CafeGuest' and 'CafeGuest_Free'. What do you do?", ["Join the strongest signal", "Ask staff for the exact network name", "Join both", "Disable your firewall"], 1, "Rogue hotspots copy trusted names."),
    ("Wi-Fi", "Which home-router step helps most?", ["Keep the default admin password", "Change default credentials and update firmware", "Hide the router in a drawer", "Disable all encryption"], 1, "Default credentials and old firmware are common weaknesses."),
    ("Mobile Security", "An app asks for contacts, microphone and location but is a simple calculator. You should...", ["Allow everything", "Deny unnecessary permissions or uninstall it", "Share it with friends", "Turn off updates"], 1, "Permissions should match function."),
    ("Mobile Security", "You lose your work phone. What should you do?", ["Wait a week", "Report it quickly so it can be locked or wiped per policy", "Post about it publicly", "Buy a new phone and say nothing"], 1, "Speed matters; the phone may hold email and MFA apps."),
    ("Mobile Security", "A text says you owe a delivery fee and includes a link. Best response?", ["Tap the link and pay", "Ignore the link and check the courier's official app/site", "Reply with card details", "Forward to friends"], 1, "Smishing imitates couriers. Go to the real service yourself."),
    ("Incident Reporting", "You clicked a suspicious link by mistake. What now?", ["Hide it", "Report it immediately", "Wait to see what happens", "Clear your history only"], 1, "Fast reporting limits harm. Honest reports are valued."),
    ("Incident Reporting", "Where should suspected phishing be reported?", ["Social media", "The official security/IT reporting channel", "A random colleague", "Nowhere"], 1, "Official channels allow analysts to block and warn others."),
    ("Incident Reporting", "Why report suspicious events quickly?", ["It looks good", "Early detection limits impact and protects others", "It is optional", "It cancels the event"], 1, "Time matters in containment.")]
QUIZ = [{"id": f"Q{i + 1:02d}", "category": c, "question": q, "options": o, "answer": a, "explanation": e} for i, (c, q, o, a, e) in enumerate(_Q)]
CAT_MODULE = {"Phishing": "M01", "Passwords": "M02", "MFA": "M03", "Social Engineering": "M04", "Safe Browsing": "M05", "Ransomware": "M08",
              "Privacy": "M10", "Wi-Fi": "M06", "Mobile Security": "M11", "Incident Reporting": "M14"}
MOD_TITLE = {m["module_id"]: m["title"] for m in MODULES}
GLOSSARY = [
    ("Cybersecurity awareness", "Helping people spot and avoid security risks.", "Behaviour-focused controls (training, nudges, reporting culture) that reduce human-driven incidents."),
    ("Cyber threat intelligence (CTI)", "Analysed information about threats that helps you decide what to do.", "Collected, processed and contextualised data about adversaries, infrastructure and vulnerabilities used for defensive decisions."),
    ("IOC (indicator of compromise)", "A clue, like a domain or file hash, that something suspicious may have happened.", "An observable artifact (IP, domain, URL, hash, sender) associated with malicious activity; it can be stale or shared."),
    ("IOA (indicator of attack)", "A clue about what someone is trying to do right now.", "Behaviour-based signal of attacker intent or technique in progress, independent of specific artifacts."),
    ("Vulnerability", "A weakness in software or configuration.", "A flaw that could be abused to violate confidentiality, integrity or availability."),
    ("CVE", "A public ID number for a known vulnerability.", "Common Vulnerabilities and Exposures: standard identifier of the form CVE-YYYY-NNNN."),
    ("CVSS", "A 0-10 score for how severe a vulnerability is in general.", "Common Vulnerability Scoring System: standardised severity rating that excludes your own asset context."),
    ("Threat feed", "A stream of threat information from a source.", "Structured or semi-structured intelligence (CSV/JSON/STIX) with varying reliability and freshness."),
    ("Threat actor", "A person or group behind malicious activity.", "A conceptual adversary profile; attribution requires strong evidence, not just shared indicators."),
    ("TTP", "The way attackers typically operate.", "Tactics, techniques and procedures describing adversary behaviour at increasing levels of detail."),
    ("MITRE ATT&CK", "A public catalogue of attacker behaviours.", "Knowledge base of adversary tactics (why) and techniques (how), used to describe and detect behaviour."),
    ("Threat enrichment", "Adding helpful context to a raw indicator.", "Augmenting an artifact with first/last seen, category, confidence, relationships and mappings."),
    ("Threat correlation", "Connecting related clues.", "Linking records through shared campaign, timing or category to form clusters; indicates relationship, not proof."),
    ("Threat hunting", "Proactively looking for signs of trouble.", "Hypothesis-driven search through telemetry for activity that automated detection missed."),
    ("SOC", "The team that watches for and responds to security events.", "Security Operations Center: people, process and tooling for monitoring, triage, investigation and response."),
    ("Strategic intelligence", "High-level trends for leaders.", "Long-term, non-technical analysis of threat landscape and business risk."),
    ("Tactical intelligence", "How attackers behave.", "TTP-focused information that guides detection and defensive design."),
    ("Operational intelligence", "Details of specific upcoming or ongoing campaigns.", "Context about campaigns, timing and targets that informs response planning."),
    ("Technical intelligence", "The raw artifacts.", "Indicators such as IPs, domains, hashes and signatures consumed by security tools.")]
IR_PHASES = [("Preparation", "Plans, contacts, logging and training ready before anything happens.", ["Reporting channel known to everyone", "Contact list current", "Backups tested"]),
             ("Detection / identification", "Spotting and confirming that an event matters.", ["Record what was seen and when", "Preserve screenshots/logs", "Classify: observation, alert, incident?"]),
             ("Containment", "Limiting spread while keeping evidence.", ["Isolate the affected account/device per policy", "Block known-bad indicators at approved controls", "Keep notes of every action"]),
             ("Eradication", "Removing the cause.", ["Remove unwanted access or software with IT", "Reset affected credentials", "Close the weakness"]),
             ("Recovery", "Returning to normal safely.", ["Restore from clean backups", "Monitor for recurrence", "Confirm services are healthy"]),
             ("Lessons learned", "Improving afterwards.", ["Hold a blameless review", "Update playbooks and training", "Track improvements"])]

# ------------------------------------------------------------------ 7. DATABASE (SQLite)
SCHEMA = """
CREATE TABLE IF NOT EXISTS sources(source_id INTEGER PRIMARY KEY, source_name TEXT UNIQUE, reliability TEXT, reliability_label TEXT);
CREATE TABLE IF NOT EXISTS threats(threat_id TEXT PRIMARY KEY, timestamp TEXT, threat_name TEXT, category TEXT, description TEXT,
  indicator_type TEXT, indicator_value TEXT, source_name TEXT, source_reliability TEXT, severity TEXT, risk_score INT, confidence_score INT,
  status TEXT, first_seen TEXT, last_seen TEXT, region TEXT, mitre_tactic TEXT, mitre_technique TEXT, mitre_technique_id TEXT, cve_id TEXT,
  campaign_id TEXT, observation_count INT, classification TEXT, behavior_observed INT);
CREATE TABLE IF NOT EXISTS indicators(indicator_id INTEGER PRIMARY KEY AUTOINCREMENT, threat_id TEXT REFERENCES threats(threat_id),
  indicator_type TEXT, indicator_value TEXT, first_seen TEXT, last_seen TEXT);
CREATE TABLE IF NOT EXISTS attack_mappings(mapping_id INTEGER PRIMARY KEY AUTOINCREMENT, threat_id TEXT REFERENCES threats(threat_id),
  tactic TEXT, technique TEXT, technique_id TEXT);
CREATE TABLE IF NOT EXISTS vulnerabilities(vulnerability_id INTEGER PRIMARY KEY AUTOINCREMENT, cve_id TEXT UNIQUE, product_category TEXT, severity TEXT,
  cvss_score REAL, published_date TEXT, patch_available INT, exploitation_status_demo TEXT, description TEXT, asset_criticality INT, exposure TEXT,
  business_impact INT, priority_score INT, priority_label TEXT);
CREATE TABLE IF NOT EXISTS alerts(alert_id TEXT PRIMARY KEY, threat_id TEXT REFERENCES threats(threat_id), timestamp TEXT, alert_type TEXT, severity TEXT,
  risk_score INT, confidence_score INT, description TEXT, status TEXT, observation_count INT, created_at TEXT);
CREATE TABLE IF NOT EXISTS analyst_notes(note_id INTEGER PRIMARY KEY AUTOINCREMENT, threat_id TEXT REFERENCES threats(threat_id), note TEXT, author TEXT, created_at TEXT);
CREATE TABLE IF NOT EXISTS awareness_modules(module_id TEXT PRIMARY KEY, title TEXT, category TEXT);
CREATE TABLE IF NOT EXISTS quiz_results(result_id INTEGER PRIMARY KEY AUTOINCREMENT, anonymous_user_id TEXT, overall_score INT, category_scores TEXT, created_at TEXT);
CREATE TABLE IF NOT EXISTS audit_log(id INTEGER PRIMARY KEY AUTOINCREMENT, ts TEXT, actor TEXT, action TEXT, target TEXT);
CREATE TABLE IF NOT EXISTS meta(k TEXT PRIMARY KEY, v TEXT);
CREATE INDEX IF NOT EXISTS ix_t_sev ON threats(severity); CREATE INDEX IF NOT EXISTS ix_t_cat ON threats(category);
CREATE INDEX IF NOT EXISTS ix_t_status ON threats(status); CREATE INDEX IF NOT EXISTS ix_t_risk ON threats(risk_score);
CREATE INDEX IF NOT EXISTS ix_t_val ON threats(indicator_value); CREATE INDEX IF NOT EXISTS ix_t_camp ON threats(campaign_id);
CREATE INDEX IF NOT EXISTS ix_i_val ON indicators(indicator_value); CREATE INDEX IF NOT EXISTS ix_a_thr ON alerts(threat_id);
CREATE INDEX IF NOT EXISTS ix_a_status ON alerts(status); CREATE INDEX IF NOT EXISTS ix_n_thr ON analyst_notes(threat_id);
"""

def init_schema(conn):
    conn.executescript(SCHEMA); conn.commit()

def build_database():
    rng = random.Random(SEED)
    vulns = generate_vulnerabilities(rng)
    threats = generate_threats(rng, vulns)
    alerts, raw_n = build_alerts(rng, threats, vulns)
    threats.to_csv(BASE / "data" / "threat_intelligence_dataset.csv", index=False)
    vulns.to_csv(BASE / "data" / "vulnerabilities.csv", index=False)
    (BASE / "awareness" / "modules.json").write_text(json.dumps(MODULES, indent=2))
    (BASE / "awareness" / "quiz_questions.json").write_text(json.dumps(QUIZ, indent=2))
    if os.path.exists(DB_PATH): os.remove(DB_PATH)
    conn = sqlite3.connect(DB_PATH); init_schema(conn)
    conn.executemany("INSERT INTO sources(source_name,reliability,reliability_label) VALUES(?,?,?)", [(k, v, REL_LABEL[v]) for k, v in SOURCES.items()])
    conn.executemany("INSERT INTO threats VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
        [(r.threat_id, r.timestamp, r.threat_name, r.threat_category, r.description, r.indicator_type, r.indicator_value, r.source_name, r.source_reliability,
          r.severity, int(r.risk_score), int(r.confidence_score), r.status, r.first_seen, r.last_seen, r.country_or_region_optional, r.mitre_tactic_optional,
          r.mitre_technique_optional, r.mitre_technique_id_optional, r.cve_id_optional, r.campaign_id, int(r.observation_count), r.classification,
          int(r.behavior_observed)) for r in threats.itertuples()])
    conn.executemany("INSERT INTO indicators(threat_id,indicator_type,indicator_value,first_seen,last_seen) VALUES(?,?,?,?,?)",
        [(r.threat_id, r.indicator_type, r.indicator_value, r.first_seen, r.last_seen) for r in threats.itertuples()])
    conn.executemany("INSERT INTO attack_mappings(threat_id,tactic,technique,technique_id) VALUES(?,?,?,?)",
        [(r.threat_id, r.mitre_tactic_optional, r.mitre_technique_optional, r.mitre_technique_id_optional) for r in threats.itertuples() if r.mitre_tactic_optional])
    conn.executemany("INSERT INTO vulnerabilities(cve_id,product_category,severity,cvss_score,published_date,patch_available,exploitation_status_demo,description,asset_criticality,exposure,business_impact,priority_score,priority_label) VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?)",
        [(r.cve_id, r.product_category, r.severity, r.cvss_score, r.published_date, int(r.patch_available), r.exploitation_status_demo, r.description,
          r.asset_criticality, r.exposure, r.business_impact, r.priority_score, r.priority_label) for r in vulns.itertuples()])
    conn.executemany("INSERT INTO alerts VALUES(?,?,?,?,?,?,?,?,?,?,?)",
        [(a["alert_id"], a["threat_id"], a["timestamp"], a["alert_type"], a["severity"], a["risk_score"], a["confidence_score"], a["description"], a["status"],
          a["observation_count"], a["timestamp"]) for a in alerts])
    conn.executemany("INSERT INTO awareness_modules VALUES(?,?,?)", [(m["module_id"], m["title"], m["category"]) for m in MODULES])
    notes = ["Reviewed - no internal sightings so far.", "Awaiting vendor confirmation.", "Duplicate of an existing cluster; merged for tracking.",
             "Low corroboration; keeping under watch.", "Escalated to email-security team for review.", "Confirmed benign shared infrastructure (demo)."]
    conn.execute("INSERT INTO analyst_notes(threat_id,note,author,created_at) VALUES(?,?,?,?)", ("THR-2026-001", "Indicator appears in multiple synthetic phishing observations.", "analyst_demo", fmt(NOW - timedelta(days=3))))
    for tid in rng.sample(threats["threat_id"].tolist()[3:], 70):
        conn.execute("INSERT INTO analyst_notes(threat_id,note,author,created_at) VALUES(?,?,?,?)", (tid, rng.choice(notes), rng.choice(["analyst_a", "analyst_b"]), fmt(NOW - timedelta(days=rng.randint(0, 30)))))
    for w in range(12):  # 12 weeks of synthetic anonymous quiz history -> awareness trend
        for _ in range(5):
            cs = {c: min(100, max(0, int(round((rng.gauss(52 + w * 2.6, 17)) / 33.34) * 33.34))) for c in CAT_MODULE}
            conn.execute("INSERT INTO quiz_results(anonymous_user_id,overall_score,category_scores,created_at) VALUES(?,?,?,?)",
                         (f"anon-{rng.randint(1000, 9999)}", int(round(sum(cs.values()) / len(cs))), json.dumps(cs), fmt(NOW - timedelta(weeks=11 - w, days=rng.randint(0, 5)))))
    conn.execute("INSERT INTO meta VALUES('raw_alert_events',?)", (str(raw_n),))
    conn.commit(); conn.close()
    return threats, vulns, alerts, raw_n

# ------------------------------------------------------------------ 8. ENGINES THAT READ THE DB
def get_conn():
    c = sqlite3.connect(DB_PATH); c.row_factory = sqlite3.Row; return c
rows = lambda cur: [dict(r) for r in cur.fetchall()]

def enrich_indicator(conn, value):
    """Enrich from LOCAL data only: no DNS, WHOIS, HTTP or any network call."""
    recs = rows(conn.execute("SELECT * FROM threats WHERE indicator_value=? ORDER BY risk_score DESC", (value,)))
    if not recs: return {"known": False}
    p = recs[0]; ids = [r["threat_id"] for r in recs]; ph = ",".join("?" * len(ids))
    camps = sorted({r["campaign_id"] for r in recs if r["campaign_id"]})
    related = []
    if camps:
        cph = ",".join("?" * len(camps))
        related = rows(conn.execute(f"SELECT DISTINCT indicator_type, indicator_value, threat_id, severity FROM threats WHERE campaign_id IN ({cph}) AND indicator_value<>? ORDER BY risk_score DESC LIMIT 15", (*camps, value)))
    return {"known": True, "primary": p, "records": recs, "indicator_type": p["indicator_type"],
            "first_seen": min(r["first_seen"] for r in recs), "last_seen": max(r["last_seen"] for r in recs),
            "categories": sorted({r["category"] for r in recs}), "total_observations": sum(r["observation_count"] for r in recs),
            "related_indicators": related, "campaigns": camps,
            "related_alerts": rows(conn.execute(f"SELECT * FROM alerts WHERE threat_id IN ({ph}) ORDER BY timestamp DESC LIMIT 10", ids)),
            "mitre": [{"tactic": r["mitre_tactic"], "technique": r["mitre_technique"], "id": r["mitre_technique_id"]} for r in recs if r["mitre_tactic"]][:3],
            "notes": rows(conn.execute(f"SELECT * FROM analyst_notes WHERE threat_id IN ({ph}) ORDER BY created_at DESC LIMIT 10", ids))}

def correlate_threats(conn, min_size=2):
    """Group records into RELATED THREAT CLUSTERS by shared campaign id. Relationship evidence, NOT attribution."""
    clusters = []
    for r in conn.execute("SELECT campaign_id, COUNT(*) n, COUNT(DISTINCT indicator_value) d, GROUP_CONCAT(DISTINCT indicator_type) types, MIN(first_seen) fs, MAX(last_seen) ls, AVG(risk_score) rs FROM threats WHERE campaign_id<>'' GROUP BY campaign_id HAVING d>=?", (min_size,)):
        clusters.append({"cluster_id": r["campaign_id"], "records": r["n"], "distinct_indicators": r["d"], "types": r["types"].split(","), "first_seen": r["fs"], "last_seen": r["ls"], "avg_risk": round(r["rs"], 1)})
    return sorted(clusters, key=lambda c: -c["avg_risk"])

def build_timeline(t):
    fs, ls = parse(t["first_seen"]), parse(t["last_seen"]); rg = random.Random(t["threat_id"]); span = ls - fs
    ev = [(fs, "First seen", f"First observed via {t['source_name']} (reliability {t['source_reliability']}).")]
    if t["observation_count"] > 1: ev.append((fs + span * 0.4, "New observations", f"{t['observation_count']} observations recorded in total."))
    d = rg.choice([-8, -5, 6, 11]); ev.append((fs + span * 0.6, "Risk increased" if d > 0 else "Risk decreased", f"Score moved by {abs(d)} points after correlation review (demo)."))
    if t["status"] != "NEW": ev.append((fs + span * 0.8, "Investigation", "Analyst triage started."))
    if t["status"] == "MONITORING": ev.append((ls, "Monitoring", "Watching for new related observations."))
    if t["status"] == "CLOSED": ev.append((ls, "Closed", "Indicator expired or resolved."))
    if t["status"] == "FALSE_POSITIVE": ev.append((ls, "Closed as false positive", "Evidence did not support the threat assessment."))
    return [{"ts": fmt(e[0]), "label": e[1], "detail": e[2]} for e in sorted(ev, key=lambda x: x[0])]

def grp(conn, col, table="threats", where=""):
    return {r[0]: r[1] for r in conn.execute(f"SELECT {col}, COUNT(*) FROM {table} {where} GROUP BY {col} ORDER BY COUNT(*) DESC")}

def build_stats(conn):
    one = lambda s, a=(): conn.execute(s, a).fetchone()[0] or 0
    total = one("SELECT COUNT(*) FROM threats")
    risks = [r[0] for r in conn.execute("SELECT risk_score FROM threats")]; confs = [r[0] for r in conn.execute("SELECT confidence_score FROM threats")]
    bins = lambda xs: {"labels": [f"{i}-{i + 9}" if i < 90 else "90-100" for i in range(0, 100, 10)], "data": [sum(1 for x in xs if min(x // 10, 9) == b) for b in range(10)]}
    tactics = [{"name": r[0], "count": r[1]} for r in conn.execute("SELECT mitre_tactic, COUNT(*) FROM threats WHERE mitre_tactic<>'' GROUP BY mitre_tactic ORDER BY 2 DESC")]
    ent = grp(conn, "classification"); ent["ALERT"] = one("SELECT COUNT(*) FROM alerts")
    return {"cards": {"total": total, "critical": one("SELECT COUNT(*) FROM threats WHERE severity='CRITICAL'"), "high": one("SELECT COUNT(*) FROM threats WHERE severity='HIGH'"),
                      "active": one("SELECT COUNT(*) FROM threats WHERE status IN ('NEW','UNDER_REVIEW','MONITORING')"), "open_inv": one("SELECT COUNT(*) FROM threats WHERE status='UNDER_REVIEW'"),
                      "avg_conf": round(one("SELECT AVG(confidence_score) FROM threats"), 1), "vulns": one("SELECT COUNT(*) FROM vulnerabilities")},
            "by_severity": {s: grp(conn, "severity").get(s, 0) for s in SEVERITIES}, "by_category": grp(conn, "category"), "by_type": grp(conn, "indicator_type"),
            "by_status": grp(conn, "status"), "tactics": tactics, "risk_bins": bins(risks), "conf_bins": bins(confs),
            "vuln_by_sev": {s: grp(conn, "severity", "vulnerabilities").get(s, 0) for s in ["LOW", "MEDIUM", "HIGH", "CRITICAL"]},
            "cat_risk": {r[0]: round(r[1], 1) for r in conn.execute("SELECT category, AVG(risk_score) FROM threats GROUP BY category ORDER BY 2 DESC")},
            "entities": {k: ent.get(k, 0) for k in ["OBSERVATION", "INDICATOR", "ALERT", "THREAT", "INCIDENT"]},
            "recent": rows(conn.execute("SELECT * FROM threats WHERE status IN ('NEW','UNDER_REVIEW','MONITORING') ORDER BY risk_score DESC, last_seen DESC LIMIT 8"))}

def build_trends(conn):
    wk = defaultdict(lambda: [0, 0])
    for r in conn.execute("SELECT first_seen, severity FROM threats"):
        d = parse(r["first_seen"]); k = (d - timedelta(days=d.weekday())).strftime("%Y-%m-%d")
        wk[k][0] += 1; wk[k][1] += r["severity"] in ("HIGH", "CRITICAL")
    ks = sorted(wk)
    return {"weeks": ks, "total": [wk[k][0] for k in ks], "high_critical": [wk[k][1] for k in ks]}

def build_attack(conn):
    tech = rows(conn.execute("SELECT mitre_tactic tactic, mitre_technique technique, mitre_technique_id id, COUNT(*) count FROM threats WHERE mitre_tactic<>'' GROUP BY 1,2,3 ORDER BY 4 DESC"))
    unm = conn.execute("SELECT COUNT(*) FROM threats WHERE mitre_tactic=''").fetchone()[0]
    return {"techniques": tech, "tactic_order": TACTIC_ORDER, "unmapped": unm, "note": "Mappings exist only where behavioural evidence and confidence >= 50 justify them."}

def score_quiz(answers):
    per = defaultdict(lambda: [0, 0]); review = []
    for q in QUIZ:
        a = answers.get(q["id"]); ok = a == q["answer"]
        per[q["category"]][1] += 1; per[q["category"]][0] += ok
        review.append({"id": q["id"], "category": q["category"], "question": q["question"], "your_answer": a, "correct": q["answer"], "options": q["options"], "is_correct": ok, "explanation": q["explanation"]})
    cat = {c: int(round(v[0] / v[1] * 100)) for c, v in per.items()}
    overall = int(round(sum(cat.values()) / len(cat)))
    level = "Needs Improvement" if overall <= 40 else "Basic Awareness" if overall <= 60 else "Good Awareness" if overall <= 80 else "Strong Awareness"
    return {"overall": overall, "level": level, "category_scores": cat, "review": review}

def generate_learning_recommendations(category_scores):
    recs = []
    for c, s in sorted(category_scores.items(), key=lambda x: x[1]):
        mid = CAT_MODULE.get(c)
        if s < 50: act, pri = f"Complete the '{MOD_TITLE.get(mid, c)}' module.", "High"
        elif s < 80: act, pri = f"Review '{MOD_TITLE.get(mid, c)}'.", "Medium"
        else: act, pri = "No immediate module required.", "None"
        recs.append({"category": c, "score": s, "priority": pri, "action": act, "module_id": mid})
    return recs

def build_executive(conn):
    one = lambda s, a=(): conn.execute(s, a).fetchone()[0] or 0
    total = one("SELECT COUNT(*) FROM threats"); active = one("SELECT COUNT(*) FROM threats WHERE status IN ('NEW','UNDER_REVIEW','MONITORING')")
    ch = one("SELECT COUNT(*) FROM threats WHERE severity IN ('CRITICAL','HIGH') AND status IN ('NEW','UNDER_REVIEW','MONITORING')")
    thr = one("SELECT COUNT(*) FROM threats WHERE classification IN ('THREAT','INCIDENT') AND status IN ('NEW','UNDER_REVIEW','MONITORING')")
    topc = [{"name": r[0], "count": r[1]} for r in conn.execute("SELECT category, COUNT(*) FROM threats WHERE severity IN ('CRITICAL','HIGH') AND status IN ('NEW','UNDER_REVIEW','MONITORING') GROUP BY 1 ORDER BY 2 DESC LIMIT 5")]
    topv = [{"name": r[0], "count": r[1]} for r in conn.execute("SELECT product_category, COUNT(*) FROM vulnerabilities WHERE priority_score>=60 GROUP BY 1 ORDER BY 2 DESC LIMIT 5")]
    p1 = one("SELECT COUNT(*) FROM vulnerabilities WHERE priority_score>=80"); nopatch = one("SELECT COUNT(*) FROM vulnerabilities WHERE priority_score>=60 AND patch_available=0")
    wk, sums = defaultdict(list), defaultdict(lambda: [0, 0])
    for r in conn.execute("SELECT overall_score, category_scores, created_at FROM quiz_results"):
        d = parse(r["created_at"]); wk[(d - timedelta(days=d.weekday())).strftime("%Y-%m-%d")].append(r["overall_score"])
        for c, v in json.loads(r["category_scores"]).items(): sums[c][0] += v; sums[c][1] += 1
    trend = [{"week": k, "score": round(sum(v) / len(v), 1)} for k, v in sorted(wk.items())]
    weak = sorted(({"name": c, "score": round(v[0] / v[1], 1)} for c, v in sums.items() if v[1]), key=lambda x: x["score"])[:3]
    share = thr / active if active else 0
    posture = "ELEVATED" if share > .25 else "MODERATE" if share > .1 else "LOW"
    prio = []
    if topc: prio.append(f"Focus detection and user warnings on {topc[0]['name'].title()} - the largest source of high-priority activity.")
    if p1: prio.append(f"Patch the {p1} vulnerabilities rated P1 first; they combine exposure, exploitation evidence and business impact.")
    if nopatch: prio.append(f"{nopatch} high-priority vulnerabilities have no patch yet - apply compensating controls.")
    if weak: prio.append(f"Run short refreshers on {weak[0]['name']} and {weak[-1]['name'] if len(weak) > 1 else weak[0]['name']}, the weakest awareness areas.")
    prio.append("Treat every indicator match as a lead for investigation, not proof of compromise.")
    narrative = (f"The demo environment tracks {total:,} intelligence records. {active:,} are still active, and {ch:,} of those are rated high or critical. "
                 f"Only {thr:,} meet the bar of a high-risk, high-confidence threat, so the overall posture reads {posture.lower()}. "
                 f"Awareness scores have {'risen' if len(trend) > 1 and trend[-1]['score'] >= trend[0]['score'] else 'moved'} from {trend[0]['score'] if trend else 0} to {trend[-1]['score'] if trend else 0} over the period.")
    return {"posture": posture, "narrative": narrative, "totals": {"total": total, "active": active, "critical_high": ch, "confirmed_class_threats": thr, "p1_vulns": p1},
            "top_categories": topc, "top_vuln_categories": topv, "awareness_trend": trend, "awareness_weak": weak, "priorities": prio}

# ------------------------------------------------------------------ 9. REST API (Flask)
app = Flask(__name__)
class ApiError(Exception):
    def __init__(self, msg, code=400): self.msg, self.code = msg, code

@app.errorhandler(ApiError)
def _api_err(e): return jsonify({"error": e.msg}), e.code
@app.errorhandler(HTTPException)
def _http_err(e): return jsonify({"error": e.description}), e.code
@app.errorhandler(Exception)
def _any_err(e):
    print("Server error:", repr(e)); return jsonify({"error": "Internal server error"}), 500

_hits = defaultdict(list)
RATE_LIMIT = 900   # requests / minute / client
@app.before_request
def _guard():
    if not request.path.startswith("/api/"): return
    ip = request.headers.get("Cf-Connecting-Ip") or request.remote_addr; now = time.time()
    _hits[ip] = [t for t in _hits[ip] if now - t < 60]
    if len(_hits[ip]) >= RATE_LIMIT: raise ApiError("Rate limit exceeded. Try again shortly.", 429)
    _hits[ip].append(now)
    public = request.path.startswith(("/api/quiz", "/api/awareness", "/api/meta"))
    role = ROLES.get(request.headers.get("X-API-Key", ""))
    g.role = role
    if not public and not role: raise ApiError("Authentication required (X-API-Key).", 401)
    if request.method in ("POST", "PUT") and not request.path.startswith("/api/quiz") and role != "analyst":
        raise ApiError("Forbidden: analyst role required for this action.", 403)

@app.after_request
def _headers(r):
    r.headers["X-Content-Type-Options"] = "nosniff"; r.headers["Referrer-Policy"] = "no-referrer"
    r.headers["Content-Security-Policy"] = ("default-src 'self'; script-src 'self' 'unsafe-inline' https://cdnjs.cloudflare.com; "
        "style-src 'self' 'unsafe-inline' https://fonts.googleapis.com; font-src https://fonts.gstatic.com; img-src 'self' data:; connect-src 'self'")
    return r

def db():
    if "db" not in g: g.db = get_conn()
    return g.db
@app.teardown_appcontext
def _close(_):
    c = g.pop("db", None)
    if c: c.close()

def audit(actor, action, target):
    db().execute("INSERT INTO audit_log(ts,actor,action,target) VALUES(?,?,?,?)", (fmt(datetime.now(timezone.utc).replace(tzinfo=None)), actor, action, target))

def int_arg(name, default=None, lo=0, hi=10**6):
    v = request.args.get(name)
    if v in (None, ""): return default
    try: n = int(v)
    except ValueError: raise ApiError(f"'{name}' must be an integer.")
    if not lo <= n <= hi: raise ApiError(f"'{name}' must be between {lo} and {hi}.")
    return n

SORTS = {"newest": "first_seen DESC", "risk": "risk_score DESC, last_seen DESC", "confidence": "confidence_score DESC, risk_score DESC", "observed": "observation_count DESC"}

@app.get("/api/meta")
def api_meta(): return jsonify({"categories": CATEGORIES, "types": ITYPES, "statuses": STATUSES, "severities": SEVERITIES[::-1], "alert_statuses": ALERT_STATUSES})

@app.get("/api/threats")
def api_threats():
    w, p = [], []
    for key, col in (("severity", "severity"), ("category", "category"), ("indicator_type", "indicator_type"), ("status", "status"), ("classification", "classification")):
        v = request.args.get(key)
        if v:
            v = v.upper()
            allowed = {"severity": SEVERITIES, "category": CATEGORIES, "indicator_type": ITYPES, "status": STATUSES, "classification": ["OBSERVATION", "INDICATOR", "THREAT", "INCIDENT"]}[key]
            if v not in allowed: raise ApiError(f"Invalid {key}.")
            w.append(f"{col}=?"); p.append(v)
    for key, col, op in (("min_risk", "risk_score", ">="), ("max_risk", "risk_score", "<="), ("min_conf", "confidence_score", ">=")):
        n = int_arg(key, None, 0, 100)
        if n is not None: w.append(f"{col}{op}?"); p.append(n)
    for key, op in (("date_from", ">="), ("date_to", "<=")):
        v = request.args.get(key)
        if v:
            if not re.fullmatch(r"\d{4}-\d{2}-\d{2}", v): raise ApiError(f"'{key}' must be YYYY-MM-DD.")
            w.append(f"substr(first_seen,1,10){op}?"); p.append(v)
    q = request.args.get("q", "").strip()
    if q:
        like = f"%{q[:100].lower()}%"; w.append("(lower(threat_id) LIKE ? OR lower(threat_name) LIKE ? OR lower(indicator_value) LIKE ?)"); p += [like] * 3
    sort = request.args.get("sort", "risk")
    if sort not in SORTS: raise ApiError("Invalid sort.")
    page, limit = int_arg("page", 1, 1, 100000), int_arg("limit", 25, 1, 100)
    ws = ("WHERE " + " AND ".join(w)) if w else ""
    total = db().execute(f"SELECT COUNT(*) FROM threats {ws}", p).fetchone()[0]
    items = rows(db().execute(f"SELECT * FROM threats {ws} ORDER BY {SORTS[sort]} LIMIT ? OFFSET ?", (*p, limit, (page - 1) * limit)))
    return jsonify({"total": total, "page": page, "limit": limit, "items": items})

@app.get("/api/threats/<tid>")
def api_threat(tid):
    t = db().execute("SELECT * FROM threats WHERE threat_id=?", (tid,)).fetchone()
    if not t: raise ApiError("Threat not found.", 404)
    t = dict(t); enr = enrich_indicator(db(), t["indicator_value"])
    same = rows(db().execute("SELECT threat_id, indicator_type, indicator_value, severity, risk_score FROM threats WHERE campaign_id=? AND campaign_id<>'' AND threat_id<>? ORDER BY risk_score DESC LIMIT 12", (t["campaign_id"], tid)))
    cve = db().execute("SELECT * FROM vulnerabilities WHERE cve_id=?", (t["cve_id"],)).fetchone() if t["cve_id"] else None
    return jsonify({"threat": t, "timeline": build_timeline(t), "related_indicators": same, "alerts": rows(db().execute("SELECT * FROM alerts WHERE threat_id=? ORDER BY timestamp DESC", (tid,))),
                    "notes": rows(db().execute("SELECT * FROM analyst_notes WHERE threat_id=? ORDER BY created_at DESC", (tid,))), "cve": dict(cve) if cve else None,
                    "recommended_actions": ACTIONS[t["category"]], "reliability_label": REL_LABEL[t["source_reliability"]], "occurrences_of_indicator": len(enr.get("records", [])),
                    "attack_note": None if t["mitre_tactic"] else "No ATT&CK mapping: insufficient behavioural evidence. An indicator alone does not prove a technique."})

@app.post("/api/threats")
def api_threat_create():
    b = request.get_json(silent=True) or {}
    name, cat, src = sanitize_text(b.get("threat_name"), 120), str(b.get("category", "")).upper(), b.get("source_name")
    v = validate_indicator(b.get("indicator_value"))
    if len(name) < 3: raise ApiError("threat_name is required (3-120 chars).")
    if cat not in CATEGORIES: raise ApiError("Invalid category.")
    if src not in SOURCES: raise ApiError("Invalid source_name.")
    if not v["valid"]: raise ApiError("Invalid indicator: " + " ".join(v["validation_notes"]))
    itype, val = TYPE_MAP[v["indicator_type"]], v["lookup_value"] if v["indicator_type"] == "EMAIL" else v["normalized_value"]
    existing = db().execute("SELECT threat_id FROM threats WHERE indicator_value=? AND category=?", (val, cat)).fetchone()
    if existing:   # duplicate observation -> merge, avoid new analyst work
        db().execute("UPDATE threats SET observation_count=observation_count+1, last_seen=? WHERE threat_id=?", (fmt(NOW), existing["threat_id"]))
        audit(g.role, "duplicate_observation_merged", existing["threat_id"]); db().commit()
        return jsonify({"threat_id": existing["threat_id"], "merged": True}), 200
    rel = SOURCES[src]; conf = calculate_confidence(rel, 1, 1, 0, False)
    sev = b.get("base_severity", "MEDIUM"); sev = sev if sev in SEV_SCORE else "MEDIUM"
    risk = calculate_threat_risk(sev, conf, NOW, 1, rel)
    n = db().execute("SELECT COUNT(*) FROM threats").fetchone()[0] + 1
    tid = f"THR-2026-{n:03d}"
    while db().execute("SELECT 1 FROM threats WHERE threat_id=?", (tid,)).fetchone(): n += 1; tid = f"THR-2026-{n:03d}"
    now = fmt(NOW)
    db().execute("INSERT INTO threats VALUES(?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)", (tid, now, name, cat, sanitize_text(b.get("description"), 500) or "Analyst-created record [SYNTHETIC / DEMO ONLY]", itype, val, src, rel,
        risk["classification"], risk["score"], conf, "NEW", now, now, "", "", "", "", "", "", 1, classify_entity(conf, risk["score"], "NEW", src, False), 0))
    db().execute("INSERT INTO indicators(threat_id,indicator_type,indicator_value,first_seen,last_seen) VALUES(?,?,?,?,?)", (tid, itype, val, now, now))
    audit(g.role, "threat_created", tid); db().commit()
    return jsonify({"threat_id": tid, "risk": risk, "confidence": conf, "merged": False}), 201

@app.put("/api/threats/<tid>")
def api_threat_update(tid):
    b = request.get_json(silent=True) or {}
    if not db().execute("SELECT 1 FROM threats WHERE threat_id=?", (tid,)).fetchone(): raise ApiError("Threat not found.", 404)
    sets, p = [], []
    if "status" in b:
        if b["status"] not in STATUSES: raise ApiError("Invalid status.")
        sets.append("status=?"); p.append(b["status"])
    if "threat_name" in b: sets.append("threat_name=?"); p.append(sanitize_text(b["threat_name"], 120))
    if "description" in b: sets.append("description=?"); p.append(sanitize_text(b["description"], 500))
    if not sets: raise ApiError("Nothing to update (status, threat_name, description).")
    db().execute(f"UPDATE threats SET {','.join(sets)} WHERE threat_id=?", (*p, tid)); audit(g.role, "threat_updated:" + ",".join(b), tid); db().commit()
    return jsonify({"threat_id": tid, "updated": list(b)})

@app.get("/api/indicators/search")
def api_search():
    q = (request.args.get("q") or "").strip()
    if not q: raise ApiError("Query parameter 'q' is required.")
    if len(q) > 2048: raise ApiError("Query too long.")
    v = validate_indicator(q)
    res = {"validation": v, "known": False}
    if v["valid"]:   # DATABASE LOOKUP ONLY - the indicator is never contacted
        e = enrich_indicator(db(), v["lookup_value"]); res.update(e)
    return jsonify(res)

@app.get("/api/dashboard/stats")
def api_stats(): return jsonify(build_stats(db()))
@app.get("/api/dashboard/trends")
def api_trends(): return jsonify(build_trends(db()))
@app.get("/api/attack")
def api_attack(): return jsonify(build_attack(db()))
@app.get("/api/attack/<tid>")
def api_attack_tech(tid):
    if not re.fullmatch(r"T\d{4}(\.\d{3})?", tid): raise ApiError("Invalid technique id.")
    return jsonify({"technique_id": tid, "threats": rows(db().execute("SELECT * FROM threats WHERE mitre_technique_id=? ORDER BY risk_score DESC LIMIT 60", (tid,)))})
@app.get("/api/clusters")
def api_clusters(): return jsonify(correlate_threats(db())[:25])
@app.get("/api/executive")
def api_exec(): return jsonify(build_executive(db()))

@app.get("/api/alerts")
def api_alerts():
    st = request.args.get("status"); w, p = "", []
    if st:
        if st.upper() not in ALERT_STATUSES: raise ApiError("Invalid status.")
        w, p = "WHERE status=?", [st.upper()]
    items = rows(db().execute(f"SELECT a.*, t.indicator_value, t.threat_name FROM alerts a JOIN threats t USING(threat_id) {w} ORDER BY a.risk_score DESC, a.timestamp DESC LIMIT ?", (*p, int_arg("limit", 100, 1, 500))))
    raw = int(db().execute("SELECT v FROM meta WHERE k='raw_alert_events'").fetchone()[0]); n = db().execute("SELECT COUNT(*) FROM alerts").fetchone()[0]
    return jsonify({"items": items, "summary": {"raw_events": raw, "alerts": n, "reduction_pct": round((1 - n / raw) * 100, 1) if raw else 0, "by_status": grp(db(), "status", "alerts")}})

@app.put("/api/alerts/<aid>/status")
def api_alert_status(aid):
    s = (request.get_json(silent=True) or {}).get("status")
    if s not in ALERT_STATUSES: raise ApiError("Invalid status.")
    if not db().execute("SELECT 1 FROM alerts WHERE alert_id=?", (aid,)).fetchone(): raise ApiError("Alert not found.", 404)
    db().execute("UPDATE alerts SET status=? WHERE alert_id=?", (s, aid)); audit(g.role, "alert_status:" + s, aid); db().commit()
    return jsonify({"alert_id": aid, "status": s})

@app.post("/api/threats/<tid>/notes")
def api_note(tid):
    if not db().execute("SELECT 1 FROM threats WHERE threat_id=?", (tid,)).fetchone(): raise ApiError("Threat not found.", 404)
    note = sanitize_text((request.get_json(silent=True) or {}).get("note"), 1000)
    if len(note) < 3: raise ApiError("Note must be 3-1000 characters.")
    db().execute("INSERT INTO analyst_notes(threat_id,note,author,created_at) VALUES(?,?,?,?)", (tid, note, "analyst_demo", fmt(datetime.now(timezone.utc).replace(tzinfo=None))))
    audit(g.role, "note_added", tid); db().commit()
    return jsonify({"threat_id": tid, "note": note}), 201

@app.get("/api/vulnerabilities")
def api_vulns():
    w, p = [], []
    sv = request.args.get("severity")
    if sv:
        if sv.upper() not in ("LOW", "MEDIUM", "HIGH", "CRITICAL"): raise ApiError("Invalid severity.")
        w.append("severity=?"); p.append(sv.upper())
    ws = ("WHERE " + " AND ".join(w)) if w else ""
    items = rows(db().execute(f"SELECT * FROM vulnerabilities {ws} ORDER BY priority_score DESC LIMIT ?", (*p, int_arg("limit", 60, 1, 200))))
    a = calculate_vulnerability_priority(9.8, 1, "ISOLATED_TEST", "NONE_OBSERVED", 1)
    b = calculate_vulnerability_priority(7.5, 5, "INTERNET_FACING", "EXPLOITED_IN_WILD_DEMO", 5)
    return jsonify({"items": items, "contrast": {"a": {"label": "CVSS 9.8 (Critical) on an isolated test asset", "label_p": a["label"], "score": a["score"]}, "b": {"label": "CVSS 7.5 (High) on an internet-facing critical system with exploitation evidence", "label_p": b["label"], "score": b["score"]}}})

@app.get("/api/awareness/modules")
def api_modules(): return jsonify({"modules": MODULES, "categories": [{"name": k, "description": v[0], "indicators": v[1], "impact": v[2], "controls": v[3], "awareness": v[4]} for k, v in CAT_INFO.items()],
                                 "glossary": [{"term": a, "simple": b, "technical": c} for a, b, c in GLOSSARY], "ir": [{"phase": a, "text": b, "checklist": c} for a, b, c in IR_PHASES]})
@app.get("/api/quiz")
def api_quiz(): return jsonify({"questions": [{k: q[k] for k in ("id", "category", "question", "options")} for q in QUIZ]})

@app.post("/api/quiz/submit")
def api_quiz_submit():
    ans = (request.get_json(silent=True) or {}).get("answers")
    if not isinstance(ans, dict) or not ans: raise ApiError("'answers' must be an object of question id -> option index.")
    clean = {}
    for k, v in ans.items():
        if k not in {q["id"] for q in QUIZ} or not isinstance(v, int) or not 0 <= v <= 3: raise ApiError(f"Invalid answer for '{k}'.")
        clean[k] = v
    r = score_quiz(clean); r["recommendations"] = generate_learning_recommendations(r["category_scores"])
    r["weakest"] = [c for c, _ in sorted(r["category_scores"].items(), key=lambda x: x[1])[:3]]
    r["disclaimer"] = "Educational score only - not a judgement of anyone's competence or job fitness."
    db().execute("INSERT INTO quiz_results(anonymous_user_id,overall_score,category_scores,created_at) VALUES(?,?,?,?)", ("anon-" + os.urandom(3).hex(), r["overall"], json.dumps(r["category_scores"]), fmt(datetime.now(timezone.utc).replace(tzinfo=None))))
    db().commit()
    return jsonify(r), 201

# ------------------------------------------------------------------ 10. FRONT-END (single page, served by Flask)
INDEX_HTML = r"""<!doctype html><html lang="en"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
<title>Aegis - Threat Intelligence & Awareness</title>
<link href="https://fonts.googleapis.com/css2?family=Manrope:wght@400;500;600;700;800&family=IBM+Plex+Mono:wght@400;500&display=swap" rel="stylesheet">
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.1/chart.umd.min.js"></script>
<style>
:root{--bg:#070B16;--bg2:#0B1226;--panel:#111A33;--panel2:#16223F;--line:#25335C;--text:#EEF3FF;--mut:#A9B6D8;--teal:#34E8C8;--amber:#FFC46B;--violet:#9D8CFF;
--CRITICAL:#FF5C7A;--HIGH:#FF9F5A;--MEDIUM:#FFD166;--LOW:#5EEAA0;--INFORMATIONAL:#7CC4FF;--r:14px}
*{box-sizing:border-box}html{scroll-behavior:smooth}
body{margin:0;background:radial-gradient(1100px 600px at 85% -10%,#1a2a5e55,transparent 60%),radial-gradient(900px 500px at -10% 110%,#0f3b4a55,transparent 60%),var(--bg);
color:var(--text);font-family:Manrope,system-ui,-apple-system,Segoe UI,sans-serif;font-size:14.5px;line-height:1.5;display:flex;min-height:100vh}
.side{width:236px;flex:0 0 236px;background:linear-gradient(180deg,#0C1430,#080D1D);border-right:1px solid var(--line);padding:22px 14px;position:sticky;top:0;height:100vh;overflow:auto}
.brand{display:flex;gap:11px;align-items:center;padding:0 8px 22px}.logo{width:38px;height:38px;border-radius:11px;background:linear-gradient(135deg,var(--teal),var(--violet));display:grid;place-items:center;color:#06101f;font-weight:800;font-size:18px}
.brand b{font-size:17px;letter-spacing:.2px;display:block}.brand span{color:var(--mut);font-size:11.5px}
.nav{display:flex;flex-direction:column;gap:3px}.nav h6{margin:16px 10px 6px;color:var(--mut);font-size:11.5px;font-weight:600}
.nav button{all:unset;cursor:pointer;display:flex;align-items:center;gap:11px;padding:10px 12px;border-radius:10px;color:#C9D4F2;font-weight:600;font-size:14px;transition:background .15s,color .15s}
.nav button i{width:8px;height:8px;border-radius:50%;background:#3a4a7c}.nav button:hover{background:#17224a;color:#fff}.nav button:focus-visible,.btn:focus-visible,select:focus-visible,input:focus-visible{outline:2px solid var(--teal);outline-offset:2px}
.nav button.on{background:linear-gradient(90deg,#1b2d5c,#14204a);color:#fff;box-shadow:inset 3px 0 0 var(--teal)}.nav button.on i{background:var(--teal);box-shadow:0 0 10px var(--teal)}
main{flex:1;min-width:0;padding:26px 30px 60px;max-width:1500px}
.top{display:flex;justify-content:space-between;align-items:flex-start;gap:16px;margin-bottom:20px;flex-wrap:wrap}
h1{margin:0;font-size:27px;font-weight:800;letter-spacing:-.4px}.sub{color:var(--mut);margin-top:4px;max-width:760px}
.pill{display:inline-flex;gap:8px;align-items:center;background:#1a2447;border:1px solid var(--line);color:#DCE5FF;padding:7px 13px;border-radius:99px;font-size:12.5px;font-weight:600}.pill:before{content:"";width:7px;height:7px;border-radius:50%;background:var(--amber)}
.panel{background:linear-gradient(180deg,var(--panel),#0F1730);border:1px solid var(--line);border-radius:var(--r);padding:18px 20px;box-shadow:0 10px 30px #00000040}
.panel h3{margin:0 0 12px;font-size:15px;font-weight:700}.grid{display:grid;gap:16px}.g2{grid-template-columns:repeat(2,1fr)}.g3{grid-template-columns:repeat(3,1fr)}.g4{grid-template-columns:repeat(4,1fr)}.g7{grid-template-columns:repeat(7,1fr)}
@media(max-width:1250px){.g7{grid-template-columns:repeat(4,1fr)}.g3{grid-template-columns:repeat(2,1fr)}}
@media(max-width:900px){body{flex-direction:column}.side{position:static;width:100%;height:auto;flex:none}.nav{flex-direction:row;flex-wrap:wrap}.nav h6{display:none}main{padding:18px 14px}.g2,.g3,.g4,.g7{grid-template-columns:1fr 1fr}}
@media(max-width:560px){.g2,.g3,.g4,.g7{grid-template-columns:1fr}}
.kpi{padding:16px 18px}.kpi small{color:var(--mut);font-weight:600;font-size:12.5px;display:block;min-height:36px}.kpi b{font-size:30px;font-weight:800;letter-spacing:-.8px;display:block;margin-top:2px}
.kpi.k-c b{color:var(--CRITICAL)}.kpi.k-h b{color:var(--HIGH)}.kpi.k-t b{color:var(--teal)}.kpi.k-v b{color:var(--violet)}
.chart{position:relative;height:250px}.chart.tall{height:300px}
.b{display:inline-block;padding:3px 10px;border-radius:99px;font-size:11.5px;font-weight:800;color:#07101F;white-space:nowrap}
.sev-CRITICAL{background:var(--CRITICAL)}.sev-HIGH{background:var(--HIGH)}.sev-MEDIUM{background:var(--MEDIUM)}.sev-LOW{background:var(--LOW)}.sev-INFORMATIONAL{background:var(--INFORMATIONAL)}
.st{display:inline-block;padding:2px 10px;border-radius:99px;font-size:11.5px;font-weight:700;border:1px solid #4a5b92;color:#D9E3FF;white-space:nowrap}
.cls-THREAT{background:#ff5c7a22;border-color:#ff5c7a;color:#FFD3DB}.cls-INCIDENT{background:#ff5c7a;border-color:#ff5c7a;color:#07101F}.cls-INDICATOR{background:#34e8c81c;border-color:#34e8c8;color:#C9FFF5}.cls-OBSERVATION{background:#7cc4ff1c;border-color:#7cc4ff;color:#D5ECFF}.cls-ALERT{background:#ffc46b22;border-color:var(--amber);color:#FFE9C4}
table{width:100%;border-collapse:collapse}th{text-align:left;color:var(--mut);font-size:12.5px;font-weight:700;padding:9px 10px;border-bottom:1px solid var(--line);white-space:nowrap}
td{padding:11px 10px;border-bottom:1px solid #1b2850;vertical-align:middle}tbody tr{transition:background .12s}tbody tr.click{cursor:pointer}tbody tr.click:hover{background:#17224a}
.mono{font-family:"IBM Plex Mono",ui-monospace,Menlo,monospace;font-size:12.5px;color:#CFFAF1}.tw{overflow-x:auto}
.ellip{max-width:250px;overflow:hidden;text-overflow:ellipsis;white-space:nowrap;display:inline-block;vertical-align:bottom}
.meter{display:inline-block;width:60px;height:7px;background:#223056;border-radius:9px;overflow:hidden;vertical-align:middle;margin-right:7px}.meter i{display:block;height:100%;border-radius:9px}
.bar{display:flex;flex-wrap:wrap;gap:10px;margin-bottom:14px;align-items:end}
.bar label{display:flex;flex-direction:column;gap:5px;color:var(--mut);font-size:12px;font-weight:600}
select,input[type=text],input[type=date],input[type=number],textarea{background:#0C1430;color:var(--text);border:1px solid #33457a;border-radius:10px;padding:9px 11px;font:inherit;min-width:130px}
input::placeholder,textarea::placeholder{color:#8392BA}textarea{width:100%;min-height:70px;resize:vertical}
.btn{all:unset;cursor:pointer;background:linear-gradient(135deg,var(--teal),#2bbfe0);color:#04121F;font-weight:800;padding:10px 18px;border-radius:10px;font-size:14px;transition:transform .12s,filter .12s}.btn:hover{filter:brightness(1.08);transform:translateY(-1px)}
.btn.ghost{background:#1a2650;color:#E2EAFF;border:1px solid #3a4c85}.btn.sm{padding:6px 12px;font-size:12.5px}.btn[disabled]{opacity:.5;cursor:not-allowed}
.chip{all:unset;cursor:pointer;background:#16224a;border:1px solid #33457a;color:#DDE6FF;padding:6px 12px;border-radius:99px;font-size:12.5px;font-weight:600}.chip:hover{border-color:var(--teal);color:#fff}
.note{background:#0e1a3c;border:1px solid #2d417c;border-left:4px solid var(--teal);border-radius:10px;padding:11px 14px;color:#DCE6FF;margin:0 0 14px}.note.warn{border-left-color:var(--amber)}
.ladder{display:grid;grid-template-columns:repeat(5,1fr);gap:0;margin-bottom:16px}.ladder>div{padding:14px 16px;border:1px solid var(--line);background:var(--panel);position:relative}
.ladder>div:first-child{border-radius:14px 0 0 14px}.ladder>div:last-child{border-radius:0 14px 14px 0}.ladder b{font-size:24px;font-weight:800;display:block}.ladder span{color:var(--mut);font-size:12.5px}.ladder em{font-style:normal;font-weight:700;font-size:12.5px;display:block}
@media(max-width:900px){.ladder{grid-template-columns:1fr 1fr}.ladder>div{border-radius:12px!important}}
.flow{display:flex;flex-wrap:wrap;gap:6px;align-items:center;margin:4px 0}.flow span{background:#16224a;border:1px solid #33457a;padding:5px 11px;border-radius:99px;font-size:12.5px;font-weight:600}.flow u{text-decoration:none;color:var(--teal);font-weight:800}
.drawer{position:fixed;top:0;right:0;width:min(720px,100%);height:100vh;background:#0A1229;border-left:1px solid var(--line);box-shadow:-20px 0 60px #000a;transform:translateX(105%);transition:transform .28s ease;z-index:50;overflow:auto;padding:24px 26px 60px}
.drawer.open{transform:none}.scrim{position:fixed;inset:0;background:#02050d99;opacity:0;pointer-events:none;transition:opacity .25s;z-index:40}.scrim.open{opacity:1;pointer-events:auto}
.x{float:right}.sec{margin:20px 0 8px;font-size:13px;font-weight:800;color:var(--teal)}
.kv{display:grid;grid-template-columns:150px 1fr;gap:6px 14px}.kv dt{color:var(--mut);font-weight:600}.kv dd{margin:0;word-break:break-word}
.score{display:grid;grid-template-columns:1fr 1fr;gap:12px}.score>div{background:#0e1838;border:1px solid var(--line);border-radius:12px;padding:12px 14px}.score b{font-size:28px;font-weight:800}.score .meter{width:100%;margin:6px 0 0;height:8px;display:block}
.tl{list-style:none;margin:0;padding:0 0 0 16px;border-left:2px solid #2d417c}.tl li{position:relative;padding:0 0 14px 14px}.tl li:before{content:"";position:absolute;left:-23px;top:5px;width:11px;height:11px;border-radius:50%;background:var(--teal);box-shadow:0 0 0 3px #0A1229}
.tl b{display:block}.tl small{color:var(--mut)}
.note-item{background:#0e1838;border:1px solid var(--line);border-radius:10px;padding:10px 12px;margin-bottom:8px}.note-item small{color:var(--mut)}
ul.clean{margin:0;padding-left:20px}ul.clean li{margin:4px 0}
.card{background:var(--panel2);border:1px solid var(--line);border-radius:14px;padding:16px;cursor:pointer;transition:border-color .15s,transform .15s}.card:hover{border-color:var(--teal);transform:translateY(-2px)}.card h4{margin:0 0 6px;font-size:15px}.card p{margin:0;color:var(--mut)}
.tag{color:var(--amber);font-weight:700;font-size:12px}
.matrix{display:grid;grid-template-columns:repeat(9,minmax(120px,1fr));gap:10px;overflow-x:auto;padding-bottom:6px}.col h5{margin:0 0 8px;font-size:12.5px;color:var(--mut);font-weight:700;min-height:34px}
.tech{all:unset;display:block;cursor:pointer;background:#16224a;border:1px solid #33457a;border-radius:10px;padding:8px 10px;margin-bottom:7px;font-size:12.5px;font-weight:600;width:calc(100% - 22px)}.tech:hover{border-color:var(--teal)}.tech small{display:block;color:var(--mut);font-weight:500}
.ring{width:150px;height:150px;border-radius:50%;display:grid;place-items:center;margin:0 auto}.ring div{width:114px;height:114px;border-radius:50%;background:var(--panel);display:grid;place-items:center;font-size:36px;font-weight:800}
.opt{display:block;width:100%;text-align:left;all:unset;box-sizing:border-box;cursor:pointer;display:block;width:100%;padding:13px 16px;margin:8px 0;border-radius:12px;border:1px solid #3a4c85;background:#121d42;font-weight:600}.opt:hover{border-color:var(--teal)}.opt.sel{border-color:var(--teal);background:#15335a;box-shadow:0 0 0 2px #34e8c855}
.prog{height:8px;background:#223056;border-radius:99px;overflow:hidden;margin:10px 0 18px}.prog i{display:block;height:100%;background:linear-gradient(90deg,var(--teal),var(--violet));transition:width .3s}
.good{color:var(--LOW)}.bad{color:var(--CRITICAL)}.load{padding:50px;text-align:center;color:var(--mut)}.err{border-color:var(--CRITICAL);color:#FFD3DB}
.rv{border-left:4px solid var(--LOW);padding:8px 12px;margin:8px 0;background:#0e1838;border-radius:8px}.rv.no{border-color:var(--CRITICAL)}
.toast{position:fixed;bottom:22px;left:50%;transform:translateX(-50%);background:#1b2a58;border:1px solid var(--teal);padding:11px 18px;border-radius:12px;z-index:99;font-weight:600}
@media(prefers-reduced-motion:reduce){*{transition:none!important}}
</style></head><body>
<aside class="side"><div class="brand"><div class="logo">A</div><div><b>Aegis</b><span>Threat intel &amp; awareness</span></div></div>
<nav class="nav" id="nav"><h6>Threat intelligence</h6>
<button data-p="overview"><i></i>Overview</button><button data-p="threats"><i></i>Threat feed</button><button data-p="search"><i></i>Indicator search</button>
<button data-p="attack"><i></i>ATT&amp;CK mapping</button><button data-p="vulns"><i></i>Vulnerabilities</button><button data-p="alerts"><i></i>Alerts &amp; SOC queue</button>
<h6>People</h6><button data-p="awareness"><i></i>Awareness center</button><button data-p="quiz"><i></i>Security quiz</button><h6>Leadership</h6><button data-p="exec"><i></i>Executive summary</button></nav></aside>
<main><div id="view"></div></main>
<div class="scrim" id="scrim"></div><aside class="drawer" id="drawer" aria-label="Threat details"></aside>
<script>
const KEY="__API_KEY__",ROLE="__ROLE__";
const $=s=>document.querySelector(s);
const esc=s=>String(s??"").replace(/[&<>"']/g,c=>({"&":"&amp;","<":"&lt;",">":"&gt;",'"':"&quot;","'":"&#39;"}[c]));
const SEV=["CRITICAL","HIGH","MEDIUM","LOW","INFORMATIONAL"],SC={CRITICAL:"#FF5C7A",HIGH:"#FF9F5A",MEDIUM:"#FFD166",LOW:"#5EEAA0",INFORMATIONAL:"#7CC4FF"};
const PAL=["#34E8C8","#9D8CFF","#FFC46B","#FF5C7A","#7CC4FF","#5EEAA0","#F78FE6","#FF9F5A","#4D8DFF","#B6C2E8"];
const sevB=s=>`<span class="b sev-${esc(s)}">${esc(s)}</span>`,stB=s=>`<span class="st">${esc(String(s).replace("_"," "))}</span>`,clsB=s=>`<span class="st cls-${esc(s)}">${esc(s)}</span>`;
const d10=s=>esc((s||"").slice(0,10)),nice=s=>esc(String(s||"").replace(/_/g," "));
const col=v=>v>80?SC.CRITICAL:v>60?SC.HIGH:v>40?SC.MEDIUM:v>20?SC.LOW:SC.INFORMATIONAL;
const meter=v=>`<span class="meter"><i style="width:${+v||0}%;background:${col(+v)}"></i></span>${esc(v)}`;
const meterC=v=>`<span class="meter"><i style="width:${+v||0}%;background:#34E8C8"></i></span>${esc(v)}`;
async function api(p,o={}){const r=await fetch(p,{...o,headers:{"Content-Type":"application/json","X-API-Key":KEY,...(o.headers||{})}});const j=await r.json().catch(()=>({}));if(!r.ok)throw new Error(j.error||("HTTP "+r.status));return j}
function toast(m){const t=document.createElement("div");t.className="toast";t.textContent=m;document.body.appendChild(t);setTimeout(()=>t.remove(),2800)}
let charts=[];const HAS_CHART=typeof Chart!=="undefined";if(HAS_CHART){Chart.defaults.color="#C3CDEA";Chart.defaults.font.family="Manrope,system-ui,sans-serif";Chart.defaults.borderColor="rgba(130,150,210,.16)"}
const mk=(id,cfg)=>{const c=document.getElementById(id);if(!c)return;if(!HAS_CHART){c.parentNode.innerHTML='<div class="load">Charts could not load (Chart.js CDN blocked). Data tables still work.</div>';return}charts.push(new Chart(c,cfg))};
const donut=(id,l,d,c)=>mk(id,{type:"doughnut",data:{labels:l,datasets:[{data:d,backgroundColor:c||PAL,borderColor:"#111A33",borderWidth:3}]},options:{maintainAspectRatio:false,cutout:"62%",plugins:{legend:{position:"right",labels:{boxWidth:10,padding:9,font:{size:11.5}}}}}});
const bars=(id,l,d,c,h)=>mk(id,{type:"bar",data:{labels:l,datasets:[{data:d,backgroundColor:c||"#34E8C8",borderRadius:6,maxBarThickness:24}]},options:{indexAxis:h?"y":"x",maintainAspectRatio:false,plugins:{legend:{display:false}},scales:{x:{grid:{display:h?true:false},ticks:{font:{size:11}}},y:{beginAtZero:true,grid:{display:h?false:true},ticks:{font:{size:11}}}}}});
const head=(t,s,extra="")=>`<div class="top"><div><h1>${t}</h1><div class="sub">${s}</div></div><div>${extra||'<span class="pill">Synthetic demo data</span>'}</div></div>`;
const panel=(t,b,cls="")=>`<div class="panel ${cls}"><h3>${t}</h3>${b}</div>`;
const cv=(id,tall)=>`<div class="chart ${tall?"tall":""}"><canvas id="${id}"></canvas></div>`;
function trows(items){return `<div class="tw"><table><thead><tr><th>Threat</th><th>Category</th><th>Indicator</th><th>Severity</th><th>Risk</th><th>Confidence</th><th>First seen</th><th>Last seen</th><th>Status</th></tr></thead><tbody>`+
(items.length?items.map(t=>`<tr class="click" data-thr="${esc(t.threat_id)}"><td style="min-width:190px"><b style="white-space:nowrap">${esc(t.threat_id)}</b><br><span style="color:var(--mut);font-size:12.5px">${esc(t.threat_name)}</span></td><td>${nice(t.category)}</td><td><span class="mono ellip" title="${esc(t.indicator_value)}">${esc(t.indicator_value)}</span><br><small style="color:var(--mut)">${nice(t.indicator_type)}</small></td><td>${sevB(t.severity)}<br>${clsB(t.classification)}</td><td>${meter(t.risk_score)}</td><td>${meterC(t.confidence_score)}</td><td>${d10(t.first_seen)}</td><td>${d10(t.last_seen)}</td><td>${stB(t.status)}</td></tr>`).join(""):`<tr><td colspan="9" class="load">No threats match these filters. Try clearing a filter.</td></tr>`)+`</tbody></table></div>`}
const pages={overview,threats,search:searchPage,attack,vulns,alerts,awareness,quiz,exec};
function go(p,arg){charts.forEach(c=>c.destroy());charts=[];document.querySelectorAll("#nav button").forEach(b=>b.classList.toggle("on",b.dataset.p===p));$("#view").innerHTML='<div class="load">Loading...</div>';pages[p](arg).catch(e=>{$("#view").innerHTML=`<div class="panel err">Something went wrong: ${esc(e.message)}</div>`});window.scrollTo(0,0)}
document.addEventListener("click",e=>{const t=e.target.closest("[data-thr],[data-p],[data-q],[data-tech],[data-mod],[data-close]");if(!t)return;
if(t.dataset.thr)openThreat(t.dataset.thr);else if(t.dataset.p&&t.closest("#nav")){go(t.dataset.p)}else if(t.dataset.q){closeDrawer();go("search",t.dataset.q)}else if(t.dataset.tech)showTech(t.dataset.tech);else if(t.dataset.mod){closeDrawer();go("awareness",t.dataset.mod)}else if(t.dataset.close)closeDrawer()});
document.addEventListener("keydown",e=>{if(e.key==="Escape")closeDrawer()});
$("#scrim").onclick=()=>closeDrawer();
function closeDrawer(){$("#drawer").classList.remove("open");$("#scrim").classList.remove("open")}
/* ---------- Overview ---------- */
async function overview(){const [s,t]=await Promise.all([api("/api/dashboard/stats"),api("/api/dashboard/trends")]),k=s.cards;
const K=[["Total threat records",k.total.toLocaleString(),""],["Critical threats",k.critical,"k-c"],["High threats",k.high,"k-h"],["Active indicators",k.active.toLocaleString(),"k-t"],["Open investigations",k.open_inv,""],["Average confidence",k.avg_conf+"%",""],["Vulnerabilities tracked",k.vulns,"k-v"]];
const E=[["OBSERVATION","Raw sighting, low confidence"],["INDICATOR","Artifact with some context"],["ALERT","A rule fired; needs triage"],["THREAT","High risk and high confidence"],["INCIDENT","Confirmed internal impact"]];
$("#view").innerHTML=head("Threat landscape","One view of external intelligence and what your people need to know. An indicator match is evidence for investigation - never automatic proof of compromise.")+
`<div class="grid g7">${K.map(x=>`<div class="panel kpi ${x[2]}"><small>${x[0]}</small><b>${x[1]}</b></div>`).join("")}</div>
<div style="height:16px"></div><div class="ladder">${E.map(x=>`<div><em><span class="st cls-${x[0]}">${x[0]}</span></em><b>${(s.entities[x[0]]||0).toLocaleString()}</b><span>${x[1]}</span></div>`).join("")}</div>
<div class="grid g3">${panel("Threats over time (weekly)",cv("c1"))}${panel("Threats by severity",cv("c2"))}${panel("Threats by category",cv("c3"))}
${panel("Indicator type distribution",cv("c4"))}${panel("Top ATT&CK tactics (evidence-based only)",cv("c5"))}${panel("Risk score distribution",cv("c6"))}
${panel("Confidence distribution",cv("c7"))}${panel("Vulnerabilities by severity",cv("c8"))}${panel("Average risk by category",cv("c9"))}${panel("Threat status distribution",cv("c10"))}</div>
<div style="height:16px"></div>${panel("Highest-risk active records",trows(s.recent))}`;
mk("c1",{type:"line",data:{labels:t.weeks.map(w=>w.slice(5)),datasets:[{label:"All records",data:t.total,borderColor:"#34E8C8",backgroundColor:"#34E8C833",fill:true,tension:.35,pointRadius:0},{label:"High + critical",data:t.high_critical,borderColor:"#FF9F5A",tension:.35,pointRadius:0}]},options:{maintainAspectRatio:false,plugins:{legend:{labels:{boxWidth:10}}},scales:{x:{grid:{display:false},ticks:{maxTicksLimit:8}},y:{beginAtZero:true}}}});
donut("c2",SEV,SEV.map(x=>s.by_severity[x]||0),SEV.map(x=>SC[x]));
bars("c3",Object.keys(s.by_category).map(x=>x.toLowerCase().replace(/\b\w/g,c=>c.toUpperCase())),Object.values(s.by_category),"#9D8CFF",true);
donut("c4",Object.keys(s.by_type).map(x=>x.replace("_"," ")),Object.values(s.by_type));
bars("c5",s.tactics.map(x=>x.name),s.tactics.map(x=>x.count),"#FFC46B",true);
bars("c6",s.risk_bins.labels,s.risk_bins.data,"#FF9F5A");bars("c7",s.conf_bins.labels,s.conf_bins.data,"#7CC4FF");
bars("c8",Object.keys(s.vuln_by_sev),Object.values(s.vuln_by_sev),["#5EEAA0","#FFD166","#FF9F5A","#FF5C7A"]);
mk("c9",{type:"radar",data:{labels:Object.keys(s.cat_risk).map(x=>x.split(" ")[0]),datasets:[{data:Object.values(s.cat_risk),borderColor:"#34E8C8",backgroundColor:"#34E8C844",pointRadius:2}]},options:{maintainAspectRatio:false,plugins:{legend:{display:false}},scales:{r:{grid:{color:"rgba(130,150,210,.2)"},angleLines:{color:"rgba(130,150,210,.2)"},ticks:{display:false},pointLabels:{font:{size:10.5}}}}}});
donut("c10",Object.keys(s.by_status).map(x=>x.replace("_"," ")),Object.values(s.by_status))}
/* ---------- Threat feed ---------- */
let F={sort:"risk",page:1};
async function threats(){const m=await api("/api/meta");const o=(a,l)=>`<option value="">${l}</option>`+a.map(x=>`<option value="${esc(x)}">${nice(x)}</option>`).join("");
$("#view").innerHTML=head("Threat feed","Filter, sort and open any record for the full investigation view.")+
`<div class="panel"><div class="bar"><label>Search<input type="text" id="f_q" placeholder="ID, name or indicator"></label>
<label>Severity<select id="f_severity">${o(m.severities,"Any")}</select></label><label>Category<select id="f_category">${o(m.categories,"Any")}</select></label>
<label>Indicator type<select id="f_indicator_type">${o(m.types,"Any")}</select></label><label>Status<select id="f_status">${o(m.statuses,"Any")}</select></label>
<label>Entity<select id="f_classification">${o(["OBSERVATION","INDICATOR","THREAT","INCIDENT"],"Any")}</select></label>
<label>Min risk<input type="number" id="f_min_risk" min="0" max="100" style="min-width:90px;width:90px"></label><label>Min confidence<input type="number" id="f_min_conf" min="0" max="100" style="min-width:90px;width:90px"></label>
<label>First seen from<input type="date" id="f_date_from"></label><label>to<input type="date" id="f_date_to"></label>
<label>Sort by<select id="f_sort"><option value="risk">Highest risk</option><option value="confidence">Highest confidence</option><option value="newest">Newest</option><option value="observed">Most observed</option></select></label>
<button class="btn" id="f_go">Apply filters</button><button class="btn ghost" id="f_clear">Clear</button></div><div id="tbl"></div><div id="pg" class="bar" style="margin:14px 0 0;justify-content:space-between"></div></div>`;
$("#f_go").onclick=()=>{F.page=1;loadT()};$("#f_clear").onclick=()=>{document.querySelectorAll("[id^=f_]").forEach(e=>{if(e.tagName!=="BUTTON")e.value=e.id==="f_sort"?"risk":""});F.page=1;loadT()};
$("#f_q").addEventListener("keydown",e=>{if(e.key==="Enter"){F.page=1;loadT()}});loadT()}
async function loadT(){const ids=["q","severity","category","indicator_type","status","classification","min_risk","min_conf","date_from","date_to","sort"];const p=new URLSearchParams({page:F.page,limit:20});
ids.forEach(i=>{const v=$("#f_"+i).value;if(v)p.set(i,v)});const r=await api("/api/threats?"+p);$("#tbl").innerHTML=trows(r.items);const pages=Math.max(1,Math.ceil(r.total/r.limit));
$("#pg").innerHTML=`<span style="color:var(--mut)">${r.total.toLocaleString()} records - page ${r.page} of ${pages}</span><span><button class="btn ghost sm" id="pv" ${r.page<=1?"disabled":""}>Previous</button> <button class="btn ghost sm" id="nx" ${r.page>=pages?"disabled":""}>Next</button></span>`;
$("#pv").onclick=()=>{F.page--;loadT()};$("#nx").onclick=()=>{F.page++;loadT()}}
/* ---------- Threat detail drawer ---------- */
async function openThreat(id){const d=$("#drawer");d.innerHTML='<div class="load">Loading...</div>';d.classList.add("open");$("#scrim").classList.add("open");
try{const r=await api("/api/threats/"+encodeURIComponent(id)),t=r.threat;
d.innerHTML=`<button class="btn ghost sm x" data-close="1">Close</button><div style="color:var(--mut);font-weight:700">${esc(t.threat_id)}</div><h1 style="font-size:22px;margin:4px 0 10px">${esc(t.threat_name)}</h1>
<div>${sevB(t.severity)} ${clsB(t.classification)} ${stB(t.status)} <span class="st">${nice(t.category)}</span></div><p>${esc(t.description)}</p>
<div class="score"><div>Risk score (how concerning)<br><b style="color:${col(t.risk_score)}">${t.risk_score}</b>/100<span class="meter"><i style="width:${t.risk_score}%;background:${col(t.risk_score)}"></i></span></div>
<div>Confidence score (how solid the evidence is)<br><b style="color:var(--teal)">${t.confidence_score}</b>/100<span class="meter"><i style="width:${t.confidence_score}%;background:var(--teal)"></i></span></div></div>
<p class="note warn" style="margin-top:12px">High risk does not mean confirmed compromise. Read risk together with confidence and source reliability.</p>
<div class="sec">Indicator (stored as data - never visited or executed)</div><dl class="kv"><dt>Type</dt><dd>${nice(t.indicator_type)}</dd><dt>Value</dt><dd class="mono">${esc(t.indicator_value)}</dd><dt>Source</dt><dd>${esc(t.source_name)} - reliability ${esc(t.source_reliability)} (${esc(r.reliability_label)})</dd>
<dt>First / last seen</dt><dd>${d10(t.first_seen)} to ${d10(t.last_seen)}</dd><dt>Observations</dt><dd>${t.observation_count}</dd><dt>Region</dt><dd>${esc(t.region||"Not provided")}</dd><dt>Campaign</dt><dd>${esc(t.campaign_id||"None")}</dd></dl>
<div class="sec">MITRE ATT&amp;CK mapping</div>${t.mitre_tactic?`<dl class="kv"><dt>Tactic</dt><dd>${esc(t.mitre_tactic)}</dd><dt>Technique</dt><dd>${esc(t.mitre_technique)} (${esc(t.mitre_technique_id)})</dd></dl>`:`<p class="note">${esc(r.attack_note)}</p>`}
${r.cve?`<div class="sec">CVE association</div><dl class="kv"><dt>CVE</dt><dd class="mono">${esc(r.cve.cve_id)}</dd><dt>CVSS</dt><dd>${r.cve.cvss_score} (${esc(r.cve.severity)})</dd><dt>Contextual priority</dt><dd>${r.cve.priority_score} - ${esc(r.cve.priority_label)}</dd><dt>Patch</dt><dd>${r.cve.patch_available?"Available":"Not yet available"}</dd></dl>`:""}
<div class="sec">Related indicators</div>${r.related_indicators.length?r.related_indicators.map(x=>`<button class="chip" data-q="${esc(x.indicator_value)}" title="Search this indicator">${esc(x.indicator_type.replace("_"," "))}: ${esc(x.indicator_value.slice(0,38))}</button> `).join(""):'<span style="color:var(--mut)">No related indicators in this synthetic dataset.</span>'}
<div class="sec">Related alerts</div>${r.alerts.length?r.alerts.map(a=>`<div class="note-item"><b>${esc(a.alert_id)}</b> ${sevB(a.severity)} ${stB(a.status)} <span class="st cls-ALERT">${a.observation_count} observation${a.observation_count>1?"s":""} merged</span><br><small>${esc(nice(a.alert_type))} - ${esc(a.description)}</small></div>`).join(""):'<span style="color:var(--mut)">No alerts fired for this record.</span>'}
<div class="sec">Timeline</div><ul class="tl">${r.timeline.map(e=>`<li><b>${esc(e.label)}</b><small>${esc(e.ts)} - ${esc(e.detail)}</small></li>`).join("")}</ul>
<div class="sec">Recommended defensive actions</div><ul class="clean">${r.recommended_actions.map(a=>`<li>${esc(a)}</li>`).join("")}</ul>
<div class="sec">Analyst workflow</div><div class="bar"><label>Status<select id="stSel">${["NEW","UNDER_REVIEW","MONITORING","CLOSED","FALSE_POSITIVE"].map(s=>`<option ${s===t.status?"selected":""} value="${s}">${nice(s)}</option>`).join("")}</select></label><button class="btn sm" id="stBtn">Update status</button></div>
<div id="notes">${r.notes.map(n=>`<div class="note-item">${esc(n.note)}<br><small>${esc(n.author)} - ${esc(n.created_at)}</small></div>`).join("")||'<span style="color:var(--mut)">No analyst notes yet.</span>'}</div>
<textarea id="noteTx" placeholder="Add an analyst note (no personal data)"></textarea><div style="margin-top:8px"><button class="btn sm" id="noteBtn">Save note</button></div>`;
$("#stBtn").onclick=async()=>{try{await api("/api/threats/"+id,{method:"PUT",body:JSON.stringify({status:$("#stSel").value})});toast("Status updated");}catch(e){toast(e.message)}};
$("#noteBtn").onclick=async()=>{const v=$("#noteTx").value;try{await api(`/api/threats/${id}/notes`,{method:"POST",body:JSON.stringify({note:v})});toast("Note saved");openThreat(id)}catch(e){toast(e.message)}}}
catch(e){d.innerHTML=`<button class="btn ghost sm x" data-close="1">Close</button><div class="panel err">${esc(e.message)}</div>`}}
/* ---------- Indicator search ---------- */
async function searchPage(pre){$("#view").innerHTML=head("Indicator search","Type an IP, domain, URL, file hash, e-mail sender or CVE ID. This is a lookup in the local demo database only - the indicator is never contacted.")+
`<div class="panel"><div class="bar" style="align-items:center"><input type="text" id="sq" placeholder="e.g. 198.51.100.25" style="flex:1;min-width:260px;font-size:15px;padding:12px 14px"><button class="btn" id="sgo">Search</button></div>
<div style="display:flex;gap:8px;flex-wrap:wrap;align-items:center"><span style="color:var(--mut)">Try:</span><button class="chip" data-q="198.51.100.25">198.51.100.25</button><button class="chip" data-q="login-check.invalid">login-check.invalid</button><button class="chip" data-q="https://login-check.invalid/verify/account">demo URL</button><button class="chip" data-q="CVE-2099-1001">CVE-2099-1001</button><button class="chip" data-q="hxxps://login-check[.]invalid/verify/account">defanged URL</button><button class="chip" data-q="999.1.1.1">invalid IP</button></div></div>
<div style="height:16px"></div>${panel("Indicator lifecycle",`<div class="flow"><span>Observed</span><u>&rarr;</u><span>Validated</span><u>&rarr;</u><span>Enriched</span><u>&rarr;</u><span>Scored</span><u>&rarr;</u><span>Investigated</span><u>&rarr;</u><span>Monitored</span><u>&rarr;</u><span>Expired / closed</span></div>`)}<div style="height:16px"></div><div id="sres"></div>`;
const run=async v=>{if(!v.trim())return;$("#sres").innerHTML='<div class="load">Looking up...</div>';try{const r=await api("/api/indicators/search?q="+encodeURIComponent(v));renderSearch(r,v)}catch(e){$("#sres").innerHTML=`<div class="panel err">${esc(e.message)}</div>`}};
$("#sgo").onclick=()=>run($("#sq").value);$("#sq").addEventListener("keydown",e=>{if(e.key==="Enter")run($("#sq").value)});if(pre){$("#sq").value=pre;run(pre)}}
function renderSearch(r,q){const v=r.validation;let h=`<div class="panel"><h3>Validation (syntax only)</h3><dl class="kv"><dt>Valid format</dt><dd>${v.valid?'<b class="good">Yes</b>':'<b class="bad">No</b>'}</dd><dt>Detected type</dt><dd>${esc(v.indicator_type)}</dd><dt>Normalized</dt><dd class="mono">${esc(v.normalized_value||"-")}</dd><dt>Notes</dt><dd>${v.validation_notes.map(esc).join("<br>")}</dd></dl></div>`;
if(v.valid&&!r.known)h+=`<div style="height:16px"></div><div class="panel"><h3>Result</h3><dl class="kv"><dt>Known in demo dataset</dt><dd><b>NO</b></dd></dl><p class="note">Not in the local dataset. That says nothing about whether it is safe - it simply has no local intelligence.</p></div>`;
if(r.known){const p=r.primary;h+=`<div style="height:16px"></div><div class="panel"><h3>Result</h3><div class="grid g3"><dl class="kv" style="grid-template-columns:140px 1fr"><dt>Indicator type</dt><dd>${nice(r.indicator_type)}</dd><dt>Known in demo dataset</dt><dd><b class="good">YES</b></dd><dt>Risk</dt><dd><b>${p.risk_score}/100</b></dd><dt>Severity</dt><dd>${sevB(p.severity)}</dd><dt>Confidence</dt><dd><b>${p.confidence_score}%</b></dd></dl>
<dl class="kv" style="grid-template-columns:140px 1fr"><dt>Category</dt><dd>${r.categories.map(nice).join(", ")}</dd><dt>First seen</dt><dd>${d10(r.first_seen)}</dd><dt>Last seen</dt><dd>${d10(r.last_seen)}</dd><dt>Status</dt><dd>${stB(p.status)}</dd><dt>Entity</dt><dd>${clsB(p.classification)}</dd></dl>
<dl class="kv" style="grid-template-columns:140px 1fr"><dt>Source</dt><dd>${esc(p.source_name)} (${esc(p.source_reliability)})</dd><dt>Observations</dt><dd>${r.total_observations}</dd><dt>ATT&amp;CK</dt><dd>${r.mitre.length?r.mitre.map(m=>esc(m.tactic+" / "+m.technique+" "+m.id)).join("<br>"):"Not mapped (insufficient behavioural evidence)"}</dd><dt>Campaign</dt><dd>${esc(r.campaigns.join(", ")||"None")}</dd></dl></div></div>
<div style="height:16px"></div><div class="grid g2">${panel("Related indicators (same campaign)",r.related_indicators.length?r.related_indicators.map(x=>`<button class="chip" data-q="${esc(x.indicator_value)}">${esc(x.indicator_type.replace("_"," "))}: ${esc(x.indicator_value.slice(0,40))}</button> `).join(""):"<span style='color:var(--mut)'>None</span>")}
${panel("Analyst notes",r.notes.length?r.notes.map(n=>`<div class="note-item">${esc(n.note)}<br><small>${esc(n.author)} - ${esc(n.created_at.slice(0,10))}</small></div>`).join(""):"<span style='color:var(--mut)'>None</span>")}</div>
<div style="height:16px"></div>${panel("Matching threat records",trows(r.records))}<p class="note warn" style="margin-top:14px">A match is a lead, not a verdict. Indicators can be stale, shared by benign infrastructure, or lack context.</p>`}
$("#sres").innerHTML=h}
/* ---------- ATT&CK ---------- */
async function attack(){const a=await api("/api/attack"),by={};a.techniques.forEach(t=>(by[t.tactic]=by[t.tactic]||[]).push(t));const mx=Math.max(...a.techniques.map(t=>t.count),1);
$("#view").innerHTML=head("MITRE ATT&CK mapping","A tactic is the adversary's goal (why); a technique is how it may be achieved. An indicator alone shows what was seen - mappings appear only when behaviour justifies them.")+
`<p class="note">${esc(a.note)} <b>${a.unmapped.toLocaleString()}</b> records are deliberately left unmapped.</p><div class="grid g2">${panel("Threats by tactic",cv("a1"))}${panel("Technique frequency",cv("a2"))}</div><div style="height:16px"></div>
${panel("Matrix view - select a technique to see its synthetic records",`<div class="matrix">${a.tactic_order.map(t=>`<div class="col"><h5>${esc(t)}</h5>${(by[t]||[]).map(x=>`<button class="tech" data-tech="${esc(x.id)}" style="box-shadow:inset 0 -4px 0 rgba(52,232,200,${(.2+.8*x.count/mx).toFixed(2)})">${esc(x.technique)}<small>${esc(x.id)} - ${x.count} records</small></button>`).join("")||'<small style="color:var(--mut)">No mapped records</small>'}</div>`).join("")}</div>`)}<div style="height:16px"></div><div id="tech"></div>`;
const tc={};a.techniques.forEach(t=>tc[t.tactic]=(tc[t.tactic]||0)+t.count);const tk=Object.entries(tc).sort((x,y)=>y[1]-x[1]);bars("a1",tk.map(x=>x[0]),tk.map(x=>x[1]),"#FFC46B",true);
bars("a2",a.techniques.slice(0,10).map(t=>t.technique),a.techniques.slice(0,10).map(t=>t.count),"#9D8CFF",true)}
async function showTech(id){const r=await api("/api/attack/"+encodeURIComponent(id));$("#tech").innerHTML=panel(`Records mapped to ${esc(id)}`,trows(r.threats));$("#tech").scrollIntoView({behavior:"smooth"})}
/* ---------- Vulnerabilities ---------- */
async function vulns(){const [r,s]=await Promise.all([api("/api/vulnerabilities?limit=40"),api("/api/dashboard/stats")]);const c=r.contrast;
$("#view").innerHTML=head("Vulnerability awareness","CVSS describes a flaw in general. Patch priority also depends on exposure, asset value, exploitation evidence and business impact. All CVE IDs here are fictional.")+
`<div class="grid g2">${panel("Why CVSS alone is not enough",`<div class="grid g2"><div class="note-item"><small>${esc(c.a.label)}</small><br><b style="font-size:26px;color:${col(c.a.score)}">${c.a.score}</b><br>Contextual priority - ${esc(c.a.label_p||"")}</div><div class="note-item"><small>${esc(c.b.label)}</small><br><b style="font-size:26px;color:${col(c.b.score)}">${c.b.score}</b><br>Contextual priority - ${esc(c.b.label_p||"")}</div></div><p class="note">Priority = CVSS + asset criticality + exposure + exploitation evidence + business context.</p>`)}
${panel("Vulnerabilities by CVSS severity",cv("v1"))}</div><div style="height:16px"></div>
${panel("Highest contextual priority",`<div class="tw"><table><thead><tr><th>CVE (synthetic)</th><th>Product</th><th>CVSS</th><th>Exposure</th><th>Exploitation (demo)</th><th>Patch</th><th>Priority</th></tr></thead><tbody>${r.items.map(v=>`<tr><td class="mono">${esc(v.cve_id)}</td><td>${esc(v.product_category)}</td><td>${sevB(v.severity)} ${v.cvss_score}</td><td>${nice(v.exposure)}</td><td>${nice(v.exploitation_status_demo)}</td><td>${v.patch_available?'<span class="good">Available</span>':'<span class="bad">None yet</span>'}</td><td>${meter(v.priority_score)}<br><small style="color:var(--mut)">${esc(v.priority_label)}</small></td></tr>`).join("")}</tbody></table></div>`)}`;
bars("v1",Object.keys(s.vuln_by_sev),Object.values(s.vuln_by_sev),["#5EEAA0","#FFD166","#FF9F5A","#FF5C7A"])}
/* ---------- Alerts ---------- */
async function alerts(){const r=await api("/api/alerts?limit=60"),m=r.summary;const S=["NEW","INVESTIGATING","MONITORING","RESOLVED","FALSE_POSITIVE"];
$("#view").innerHTML=head("Alerts and SOC queue","Repeated events are merged into one alert with an observation count, so analysts see one item instead of a flood (alert fatigue).")+
`<div class="grid g4"><div class="panel kpi"><small>Raw events received</small><b>${m.raw_events.toLocaleString()}</b></div><div class="panel kpi k-t"><small>Alerts after correlation</small><b>${m.alerts.toLocaleString()}</b></div><div class="panel kpi k-v"><small>Noise removed</small><b>${m.reduction_pct}%</b></div><div class="panel kpi"><small>New alerts waiting</small><b>${m.by_status.NEW||0}</b></div></div><div style="height:16px"></div>
${panel("Tier 1 analyst workflow",`<div class="flow"><span>Threat feed</span><u>&rarr;</u><span>IOC detected</span><u>&rarr;</u><span>Validate</span><u>&rarr;</u><span>Enrich</span><u>&rarr;</u><span>Risk + confidence</span><u>&rarr;</u><span>Alert</span><u>&rarr;</u><span>Triage</span><u>&rarr;</u><span>Correlate</span><u>&rarr;</u><span>Escalate / monitor / resolve / false positive</span><u>&rarr;</u><span>Document</span></div>`)}<div style="height:16px"></div>
${panel("Queue (highest risk first)",`<div class="tw"><table><thead><tr><th>Alert</th><th>Rule</th><th>Threat</th><th>Severity</th><th>Risk / conf.</th><th>Merged events</th><th>When</th><th>Status</th></tr></thead><tbody>${r.items.map(a=>`<tr><td><b>${esc(a.alert_id)}</b></td><td>${nice(a.alert_type)}</td><td><button class="chip" data-thr="${esc(a.threat_id)}">${esc(a.threat_id)}</button><br><span class="mono ellip" title="${esc(a.indicator_value)}">${esc(a.indicator_value)}</span></td><td>${sevB(a.severity)}</td><td>${a.risk_score} / ${a.confidence_score}</td><td><span class="st cls-ALERT">${a.observation_count}</span></td><td>${esc(a.timestamp.slice(0,16))}</td><td><select data-aid="${esc(a.alert_id)}" class="asel" style="min-width:120px">${S.map(s=>`<option value="${s}" ${s===a.status?"selected":""}>${nice(s)}</option>`).join("")}</select></td></tr>`).join("")}</tbody></table></div>`)}`;
document.querySelectorAll(".asel").forEach(s=>s.onchange=async()=>{try{await api(`/api/alerts/${s.dataset.aid}/status`,{method:"PUT",body:JSON.stringify({status:s.value})});toast("Alert status updated")}catch(e){toast(e.message)}})}
/* ---------- Awareness ---------- */
let AW=null,awTab="modules";
async function awareness(sel){if(!AW)AW=await api("/api/awareness/modules");const T=[["modules","Learning modules"],["cats","Threat categories"],["ir","Incident response"],["gloss","Concepts glossary"]];
let body="";
if(awTab==="modules"){const cur=sel&&AW.modules.find(m=>m.module_id===sel);
body=cur?`<button class="btn ghost sm" id="back">Back to all modules</button><div class="panel" style="margin-top:14px"><h3 style="font-size:22px">${esc(cur.title)}</h3><p><b>What is it?</b> ${esc(cur.what)}</p><p><b>Why it matters:</b> ${esc(cur.why)}</p><div class="grid g3"><div><div class="sec">Warning signs</div><ul class="clean">${cur.warning_signs.map(x=>`<li>${esc(x)}</li>`).join("")}</ul></div><div><div class="sec">Safe practices</div><ul class="clean">${cur.safe_practices.map(x=>`<li>${esc(x)}</li>`).join("")}</ul></div><div><div class="sec">If something happens</div><ul class="clean">${cur.if_something_happens.map(x=>`<li>${esc(x)}</li>`).join("")}</ul></div></div></div>`
:`<div class="grid g3">${AW.modules.map(m=>`<div class="card" data-mod="${esc(m.module_id)}"><span class="tag">${esc(m.category)}</span><h4>${esc(m.title)}</h4><p>${esc(m.what)}</p></div>`).join("")}</div>`}
else if(awTab==="cats")body=`<div class="grid g2">${AW.categories.map(c=>panel(esc(c.name),`<p>${esc(c.description)}</p><dl class="kv"><dt>Common signs</dt><dd>${esc(c.indicators)}</dd><dt>Impact</dt><dd>${esc(c.impact)}</dd><dt>Defences</dt><dd>${esc(c.controls)}</dd><dt>For users</dt><dd>${esc(c.awareness)}</dd></dl>`)).join("")}</div>`;
else if(awTab==="ir")body=`<p class="note">Frameworks use slightly different names for these phases; the ideas are the same. This checklist is educational and contains no destructive actions.</p><div class="grid g3">${AW.ir.map((p,i)=>panel(`${i+1}. ${esc(p.phase)}`,`<p>${esc(p.text)}</p><ul class="clean">${p.checklist.map(x=>`<li>${esc(x)}</li>`).join("")}</ul>`)).join("")}</div>`;
else body=`<div class="grid g2">${AW.glossary.map(g=>panel(esc(g.term),`<p><b>Simple:</b> ${esc(g.simple)}</p><p style="color:var(--mut)"><b>Technical:</b> ${esc(g.technical)}</p>`)).join("")}</div>`;
$("#view").innerHTML=head("Cybersecurity awareness center","Human defence plus technical intelligence: short lessons on what to look for and what to do.",'<button class="btn" id="toq">Take the quiz</button>')+
`<div class="bar">${T.map(t=>`<button class="chip" data-tab="${t[0]}" style="${t[0]===awTab?"border-color:var(--teal);color:#fff;background:#183a55":""}">${t[1]}</button>`).join("")}</div>${body}`;
document.querySelectorAll("[data-tab]").forEach(b=>b.onclick=()=>{awTab=b.dataset.tab;awareness()});$("#toq").onclick=()=>go("quiz");const bk=$("#back");if(bk)bk.onclick=()=>awareness()}
/* ---------- Quiz ---------- */
let Q={qs:[],i:0,ans:{}};
async function quiz(){const r=await api("/api/quiz");Q={qs:r.questions,i:0,ans:{}};
$("#view").innerHTML=head("Security awareness quiz",`${Q.qs.length} questions across 10 topics. Your result is an educational score, not a judgement of competence.`,'<span class="pill">Anonymous</span>')+`<div class="panel" id="qbox" style="max-width:820px"></div>`;showQ()}
function showQ(){const q=Q.qs[Q.i],b=$("#qbox");b.innerHTML=`<div style="color:var(--mut);font-weight:700">${esc(q.category)} - question ${Q.i+1} of ${Q.qs.length}</div><div class="prog"><i style="width:${Q.i/Q.qs.length*100}%"></i></div><h3 style="font-size:19px;line-height:1.4">${esc(q.question)}</h3>
${q.options.map((o,i)=>`<button class="opt ${Q.ans[q.id]===i?"sel":""}" data-o="${i}">${"ABCD"[i]}. ${esc(o)}</button>`).join("")}
<div class="bar" style="margin:16px 0 0;justify-content:space-between"><button class="btn ghost sm" id="qp" ${Q.i===0?"disabled":""}>Back</button><button class="btn" id="qn" ${Q.ans[q.id]===undefined?"disabled":""}>${Q.i===Q.qs.length-1?"Finish and score":"Next"}</button></div>`;
b.querySelectorAll(".opt").forEach(o=>o.onclick=()=>{Q.ans[q.id]=+o.dataset.o;showQ()});$("#qp").onclick=()=>{Q.i--;showQ()};
$("#qn").onclick=async()=>{if(Q.i<Q.qs.length-1){Q.i++;showQ()}else{try{renderResult(await api("/api/quiz/submit",{method:"POST",body:JSON.stringify({answers:Q.ans})}))}catch(e){toast(e.message)}}}}
function renderResult(r){const cl=r.overall>80?"#5EEAA0":r.overall>60?"#34E8C8":r.overall>40?"#FFD166":"#FF5C7A";
$("#qbox").style.maxWidth="none";$("#qbox").innerHTML=`<div class="grid g3"><div><div class="ring" style="background:conic-gradient(${cl} ${r.overall}%,#223056 0)"><div>${r.overall}</div></div><h3 style="text-align:center;margin-top:12px">${esc(r.level)}</h3><p style="text-align:center;color:var(--mut)">Overall awareness score</p></div>
<div style="grid-column:span 2"><h3>Category scores</h3>${Object.entries(r.category_scores).map(([c,s])=>`<div style="display:grid;grid-template-columns:150px 1fr 44px;gap:10px;align-items:center;margin:7px 0"><span>${esc(c)}</span><span class="meter" style="width:100%;margin:0"><i style="width:${s}%;background:${s>=80?"#5EEAA0":s>=50?"#FFD166":"#FF5C7A"}"></i></span><b>${s}%</b></div>`).join("")}</div></div>
<div class="sec">Recommended learning (weakest areas first)</div><div class="grid g2">${r.recommendations.map(x=>`<div class="note-item"><b>${esc(x.category)} - ${x.score}%</b> <span class="st">${esc(x.priority)} priority</span><br>${esc(x.action)} ${x.priority!=="None"?`<button class="chip" data-mod="${esc(x.module_id)}">Open module</button>`:""}</div>`).join("")}</div>
<div class="sec">Answer review</div>${r.review.map(x=>`<div class="rv ${x.is_correct?"":"no"}"><b>${esc(x.question)}</b><br>${x.is_correct?'<span class="good">Correct.</span>':`<span class="bad">Your answer:</span> ${esc(x.options[x.your_answer])||"-"}<br><span class="good">Correct answer:</span> ${esc(x.options[x.correct])}`}<br><small style="color:var(--mut)">${esc(x.explanation)}</small></div>`).join("")}
<p class="note" style="margin-top:14px">${esc(r.disclaimer)}</p><button class="btn" id="again">Retake quiz</button>`;$("#again").onclick=()=>go("quiz");window.scrollTo(0,0)}
/* ---------- Executive ---------- */
async function exec(){const e=await api("/api/executive"),t=e.totals,pc={ELEVATED:"#FF9F5A",MODERATE:"#FFD166",LOW:"#5EEAA0"}[e.posture];
$("#view").innerHTML=head("Executive cybersecurity summary","A plain-language view for non-technical stakeholders.")+
`<div class="panel" style="border-color:${pc}"><div style="display:flex;gap:18px;align-items:center;flex-wrap:wrap"><div style="min-width:150px"><div style="color:var(--mut);font-weight:700">Overall posture</div><div style="font-size:34px;font-weight:800;color:${pc}">${esc(e.posture)}</div></div><p style="flex:1;min-width:260px;margin:0;font-size:15.5px">${esc(e.narrative)}</p></div></div><div style="height:16px"></div>
<div class="grid g4"><div class="panel kpi"><small>Intelligence records</small><b>${t.total.toLocaleString()}</b></div><div class="panel kpi k-h"><small>Active high / critical</small><b>${t.critical_high}</b></div><div class="panel kpi k-c"><small>High-confidence threats</small><b>${t.confirmed_class_threats}</b></div><div class="panel kpi k-v"><small>Vulnerabilities to patch first (P1)</small><b>${t.p1_vulns}</b></div></div><div style="height:16px"></div>
<div class="grid g3">${panel("Top threat categories (high / critical)",cv("e1"))}${panel("Top vulnerability areas",cv("e2"))}${panel("Awareness score trend",cv("e3"))}</div><div style="height:16px"></div>
<div class="grid g2">${panel("Weakest awareness areas",e.awareness_weak.map(w=>`<div style="display:grid;grid-template-columns:150px 1fr 50px;gap:10px;align-items:center;margin:8px 0"><span>${esc(w.name)}</span><span class="meter" style="width:100%;margin:0"><i style="width:${w.score}%;background:#FFD166"></i></span><b>${w.score}</b></div>`).join(""))}
${panel("Recommended defensive priorities",`<ol class="clean">${e.priorities.map(p=>`<li>${esc(p)}</li>`).join("")}</ol>`)}</div>`;
bars("e1",e.top_categories.map(x=>x.name.split(" ")[0]),e.top_categories.map(x=>x.count),"#FF9F5A",true);bars("e2",e.top_vuln_categories.map(x=>x.name),e.top_vuln_categories.map(x=>x.count),"#9D8CFF",true);
mk("e3",{type:"line",data:{labels:e.awareness_trend.map(x=>x.week.slice(5)),datasets:[{data:e.awareness_trend.map(x=>x.score),borderColor:"#34E8C8",backgroundColor:"#34E8C833",fill:true,tension:.35}]},options:{maintainAspectRatio:false,plugins:{legend:{display:false}},scales:{y:{min:0,max:100},x:{grid:{display:false}}}}})}
if(ROLE==="viewer")document.title+=" (read-only)";
go("overview");
</script></body></html>"""

@app.get("/")
def index():
    role_key, role = (VIEWER_KEY, "viewer") if request.headers.get("Cf-Ray") else (ANALYST_KEY, "analyst")   # public tunnel = read-only
    return Response(INDEX_HTML.replace("__API_KEY__", role_key).replace("__ROLE__", role), mimetype="text/html")

# ------------------------------------------------------------------ 11. BUILD DATA + RUN AUTOMATED TESTS
print("Step 1/4  Generating synthetic data and building the database ...")
THREATS_DF, VULNS_DF, ALERTS_LIST, RAW_EVENTS = build_database()
print(f"          {len(THREATS_DF):,} threat records | {len(VULNS_DF)} vulnerabilities | {len(ALERTS_LIST):,} correlated alerts (from {RAW_EVENTS:,} raw events)")

def run_tests():
    C = app.test_client(); H = {"X-API-Key": ANALYST_KEY}; V = {"X-API-Key": VIEWER_KEY}
    res, T = [], []
    def t(tid, scenario, inp, exp, fn): T.append((tid, scenario, inp, exp, fn))
    vv = validate_indicator
    t(1, "Valid IPv4", "198.51.100.25", "valid, IPV4", lambda: (vv("198.51.100.25")["indicator_type"] == "IPV4", "IPV4"))
    t(2, "Invalid IPv4", "999.1.1.1", "invalid", lambda: (not vv("999.1.1.1")["valid"], "rejected"))
    t(3, "Valid IPv6", "2001:db8::1", "valid, IPV6", lambda: (vv("2001:db8::1")["indicator_type"] == "IPV6", vv("2001:db8::1")["indicator_type"]))
    t(4, "Valid domain", "login-check.invalid", "valid, DOMAIN", lambda: (vv("login-check.invalid")["indicator_type"] == "DOMAIN", "DOMAIN"))
    t(5, "Invalid domain", "bad_domain..com", "invalid", lambda: (not vv("bad_domain..com")["valid"], "rejected"))
    t(6, "Valid URL", "https://login-check.invalid/verify", "valid, URL", lambda: (vv("https://login-check.invalid/verify")["indicator_type"] == "URL", "URL"))
    t(7, "MD5-format hash", "32 hex chars", "MD5", lambda: (vv("d41d8cd98f00b204e9800998ecf8427e")["indicator_type"] == "MD5", "MD5"))
    t(8, "SHA1-format hash", "40 hex chars", "SHA1", lambda: (vv("da39a3ee5e6b4b0d3255bfef95601890afd80709")["indicator_type"] == "SHA1", "SHA1"))
    t(9, "SHA256-format hash", "64 hex chars", "SHA256", lambda: (vv("a" * 64)["indicator_type"] == "SHA256", "SHA256"))
    t(10, "Valid CVE format", "CVE-2099-1001", "CVE_ID", lambda: (vv("cve-2099-1001")["normalized_value"] == "CVE-2099-1001", "CVE-2099-1001"))
    t(11, "Invalid CVE format", "CVE-99-1", "invalid", lambda: (not vv("CVE-99-1")["valid"], "rejected"))
    def threat_create():
        r = C.post("/api/threats", json={"threat_name": "Unit Test Record", "category": "PHISHING", "indicator_value": "unit-test-case.example.com", "source_name": "Internal SOC"}, headers=H)
        return r.status_code == 201, f"HTTP {r.status_code} {r.get_json().get('threat_id')}"
    t(12, "Threat creation", "POST valid body", "201", threat_create)
    r1 = calculate_threat_risk("HIGH", 80, NOW, 20, "A", 2, 30)
    t(13, "Risk calculation", "HIGH, conf 80, fresh", "0-100, band matches", lambda: (0 <= r1["score"] <= 100 and r1["classification"] == classify_score(r1["score"]), f"{r1['score']} {r1['classification']}"))
    c1, c2 = calculate_confidence("A", 3, 30, 1, True), calculate_confidence("D", 1, 1, 100, False)
    t(14, "Confidence calculation", "strong vs weak evidence", "strong > weak", lambda: (c1 > c2, f"{c1} > {c2}"))
    t(15, "Source reliability", "A vs D source", "A scores higher", lambda: (calculate_threat_risk("MEDIUM", 60, NOW, 5, "A")["score"] > calculate_threat_risk("MEDIUM", 60, NOW, 5, "D")["score"], "A > D"))
    def enr():
        c = get_conn(); e = enrich_indicator(c, "login-check.invalid"); c.close()
        return e["known"] and len(e["related_indicators"]) >= 1, f"known, {len(e['related_indicators'])} related"
    t(16, "IOC enrichment", "login-check.invalid", "known + related", enr)
    def srch():
        j = C.get("/api/indicators/search?q=198.51.100.25", headers=H).get_json(); return j["known"] and j["primary"]["risk_score"] == 78, f"risk {j['primary']['risk_score']}"
    t(17, "Indicator search", "198.51.100.25", "known, risk 78", srch)
    def corr():
        c = get_conn(); cl = correlate_threats(c); c.close(); return any(x["cluster_id"] == "CMP-0001" for x in cl), f"{len(cl)} clusters"
    t(18, "Threat correlation", "shared campaign", "CMP-0001 cluster", corr)
    def dup():
        r = C.post("/api/threats", json={"threat_name": "Unit Test Record", "category": "PHISHING", "indicator_value": "unit-test-case.example.com", "source_name": "Internal SOC"}, headers=H)
        return r.status_code == 200 and r.get_json()["merged"], f"HTTP {r.status_code} merged"
    t(19, "Duplicate observation", "same indicator again", "merged, 200", dup)
    row = {"threat_id": "X", "status": "NEW", "risk_score": 90, "confidence_score": 80, "observation_count": 3, "severity": "CRITICAL"}
    t(20, "Alert generation", "risk 90 / conf 80", "HIGH_RISK_HIGH_CONFIDENCE", lambda: (generate_threat_alert(row)["alert_type"] == "HIGH_RISK_HIGH_CONFIDENCE", "alert fired"))
    ev = [dict(generate_threat_alert(row), timestamp=fmt(NOW + timedelta(seconds=i))) for i in range(100)]
    t(21, "Alert correlation", "100 events / 100 s", "1 alert, count 100", lambda: (len(correlate_alerts(ev)) == 1 and correlate_alerts(ev)[0]["observation_count"] == 100, "1 alert x100"))
    def ast():
        aid = C.get("/api/alerts?limit=1", headers=H).get_json()["items"][0]["alert_id"]
        r = C.put(f"/api/alerts/{aid}/status", json={"status": "INVESTIGATING"}, headers=H); return r.status_code == 200, f"HTTP {r.status_code}"
    t(22, "Alert status update", "PUT INVESTIGATING", "200", ast)
    def note():
        r = C.post("/api/threats/THR-2026-001/notes", json={"note": "[unit-test] <b>hello</b>"}, headers=H); return r.status_code == 201 and "<b>" not in r.get_json()["note"], "saved + sanitised"
    t(23, "Analyst notes", "POST note with HTML", "201, tags stripped", note)
    t(24, "ATT&CK mapping", "phishing with behaviour vs without", "mapped / None", lambda: (map_attack("PHISHING", "DOMAIN", True, 80)[2] == "T1566" and map_attack("PHISHING", "DOMAIN", False, 80) is None, "T1566 / None"))
    pa, pb = calculate_vulnerability_priority(9.8, 1, "ISOLATED_TEST", "NONE_OBSERVED", 1), calculate_vulnerability_priority(7.5, 5, "INTERNET_FACING", "EXPLOITED_IN_WILD_DEMO", 5)
    t(25, "Vulnerability scoring", "CVSS 9.8 isolated vs 7.5 exposed", "exposed ranks higher", lambda: (pb["score"] > pa["score"], f"{pb['score']} > {pa['score']}"))
    t(26, "Dashboard statistics", "GET stats", "total = 2000+", lambda: (C.get("/api/dashboard/stats", headers=H).get_json()["cards"]["total"] >= 2000, "OK"))
    t(27, "Severity filtering", "severity=HIGH", "all HIGH", lambda: (all(i["severity"] == "HIGH" for i in C.get("/api/threats?severity=HIGH", headers=H).get_json()["items"]), "all HIGH"))
    t(28, "Category filtering", "category=RANSOMWARE", "all RANSOMWARE", lambda: (all(i["category"] == "RANSOMWARE" for i in C.get("/api/threats?category=RANSOMWARE", headers=H).get_json()["items"]), "all RANSOMWARE"))
    def srt():
        x = [i["risk_score"] for i in C.get("/api/threats?sort=risk&limit=30", headers=H).get_json()["items"]]; return x == sorted(x, reverse=True), "descending"
    t(29, "Threat sorting", "sort=risk", "descending", srt)
    t(30, "Awareness module retrieval", "GET modules", "15 modules", lambda: (len(C.get("/api/awareness/modules").get_json()["modules"]) == 15, "15"))
    t(31, "Quiz scoring", "all correct", "100 / Strong", lambda: (score_quiz({q["id"]: q["answer"] for q in QUIZ})["overall"] == 100, "100"))
    t(32, "Learning recommendation", "40 / 55 / 90", "Complete / Review / None", lambda: ([x["priority"] for x in generate_learning_recommendations({"Phishing": 40, "Social Engineering": 55, "Passwords": 90})] == ["High", "Medium", "None"], "High,Medium,None"))
    def empty():
        c = sqlite3.connect(":memory:"); c.row_factory = sqlite3.Row; init_schema(c); s = build_stats(c); return s["cards"]["total"] == 0 and s["cards"]["avg_conf"] == 0, "zeros, no crash"
    t(33, "Empty dataset", "empty DB", "zero stats", empty)
    t(34, "API validation", "bad severity, no key, viewer write", "400 / 401 / 403", lambda: ((C.get("/api/threats?severity=BAD", headers=H).status_code, C.get("/api/threats").status_code, C.post("/api/threats/THR-2026-001/notes", json={"note": "xxx"}, headers=V).status_code) == (400, 401, 403), "400/401/403"))
    def persist():
        c = sqlite3.connect(DB_PATH); n = c.execute("SELECT COUNT(*) FROM analyst_notes WHERE note LIKE '[unit-test]%'").fetchone()[0]; c.close(); return n >= 1, f"{n} note(s) read from fresh connection"
    t(35, "Database persistence", "reopen DB", "note exists", persist)
    t(36, "Defanged indicator handling", "hxxps://evil[.]invalid/x", "re-fanged, valid URL", lambda: (vv("hxxps://login-check[.]invalid/x")["normalized_value"] == "https://login-check.invalid/x", "re-fanged"))
    for tid, sc, inp, exp, fn in T:
        try: ok, act = fn()
        except Exception as ex: ok, act = False, repr(ex)
        res.append({"Test ID": tid, "Scenario": sc, "Input": inp, "Expected": exp, "Actual": act, "Result": "PASS" if ok else "FAIL"})
    c = sqlite3.connect(DB_PATH)   # clean up test artefacts
    c.execute("DELETE FROM analyst_notes WHERE note LIKE '[unit-test]%'"); c.execute("DELETE FROM indicators WHERE indicator_value='unit-test-case.example.com'")
    c.execute("DELETE FROM threats WHERE indicator_value='unit-test-case.example.com'"); c.execute("DELETE FROM audit_log"); c.commit(); c.close()
    return pd.DataFrame(res)

print("Step 2/4  Running automated tests ...")
TEST_DF = run_tests()
TEST_DF.to_csv(BASE / "reports" / "test_results.csv", index=False)
try:
    from IPython.display import display, HTML
    display(TEST_DF.style.hide(axis="index").map(lambda v: "color:#0a7d4f;font-weight:bold" if v == "PASS" else ("color:#c0182f;font-weight:bold" if v == "FAIL" else ""), subset=["Result"]))
except Exception:
    print(TEST_DF.to_string(index=False))
print(f"          {(TEST_DF.Result == 'PASS').sum()}/{len(TEST_DF)} tests passed")

# project support files
(BASE / "requirements.txt").write_text("flask>=3.0\npandas>=2.0\n")
(BASE / ".env.example").write_text("TI_ANALYST_KEY=change-me-analyst\nTI_VIEWER_KEY=change-me-viewer\n")
(BASE / ".gitignore").write_text("__pycache__/\n*.pyc\n.env\n*.db\ncloudflared\n.ipynb_checkpoints/\n")
(BASE / "README.md").write_text("# Cybersecurity Awareness & Threat Intelligence Dashboard\n\nDefensive, educational project using synthetic data only.\n\n"
    "This project is designed exclusively for defensive cybersecurity education, threat-intelligence analysis, and security awareness. "
    "It does not execute, deploy, or interact with malicious payloads or unauthorized systems.\n")

# ------------------------------------------------------------------ 12. LAUNCH SERVER + LINKS
print("Step 3/4  Starting the dashboard server ...")
if globals().get("_ti_server"):
    try: _ti_server.shutdown()
    except Exception: pass
if globals().get("_cf_proc"):
    try: _cf_proc.terminate()
    except Exception: pass
_ti_server = make_server("0.0.0.0", PORT, app, threaded=True)
threading.Thread(target=_ti_server.serve_forever, daemon=True).start()
time.sleep(1)

def start_public_tunnel(port, timeout=45):
    """Optional public (read-only) link through a Cloudflare quick tunnel. Falls back silently if unavailable."""
    global _cf_proc
    try:
        exe = "./cloudflared"
        if not os.path.exists(exe):
            subprocess.run(["wget", "-q", "-O", exe, "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"], timeout=90, check=True)
            os.chmod(exe, 0o755)
        _cf_proc = subprocess.Popen([exe, "tunnel", "--url", f"http://127.0.0.1:{port}", "--no-autoupdate"], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        qq = queue.Queue()
        threading.Thread(target=lambda: [qq.put(l) for l in _cf_proc.stdout], daemon=True).start()
        t0 = time.time()
        while time.time() - t0 < timeout:
            try: line = qq.get(timeout=1)
            except queue.Empty: continue
            m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", line)
            if m: return m.group(0)
    except Exception as ex:
        print("          (public tunnel unavailable:", type(ex).__name__, ")")
    return None

print("Step 4/4  Creating shareable links ...")
proxy_url = None
try:
    from google.colab import output
    proxy_url = output.eval_js(f"google.colab.kernel.proxyPort({PORT})")
except Exception:
    pass
public_url = start_public_tunnel(PORT)
local_url = f"http://127.0.0.1:{PORT}"
try:
    from IPython.display import display, HTML
    card = lambda title, url, desc: (f'<div style="margin:10px 0;padding:16px 20px;border-radius:14px;background:#111A33;border:1px solid #25335C;font-family:Manrope,system-ui,sans-serif;color:#EEF3FF">'
        f'<div style="font-weight:800;font-size:15px">{title}</div><div style="color:#A9B6D8;margin:4px 0 10px;font-size:13px">{desc}</div>'
        f'<a href="{url}" target="_blank" style="background:linear-gradient(135deg,#34E8C8,#2bbfe0);color:#04121F;font-weight:800;padding:9px 16px;border-radius:10px;text-decoration:none;display:inline-block">Open dashboard</a>'
        f'<div style="margin-top:10px;font-family:monospace;font-size:12.5px;color:#CFFAF1;word-break:break-all">{url}</div></div>')
    if public_url: display(HTML(card("Public link (read-only viewer)", public_url, "Share this temporary link with anyone. It stays live while this Colab runtime is running.")))
    if proxy_url: display(HTML(card("Colab link (full analyst access)", proxy_url, "Works for you while signed in to the same Google account.")))
    if not (public_url or proxy_url): display(HTML(card("Local link", local_url, "Running inside this machine.")))
    if proxy_url:
        try: output.serve_kernel_port_as_iframe(PORT, height=820)
        except Exception: pass
except Exception:
    print("Dashboard:", public_url or proxy_url or local_url)
print("\nKeep this Colab runtime running to keep the dashboard online. Re-run the cell to rebuild everything.")
print("Reminder: all data is synthetic. Indicators are analysed as data and never contacted.")

Step 1/4  Generating synthetic data and building the database ...
          2,000 threat records | 120 vulnerabilities | 813 correlated alerts (from 11,475 raw events)
Step 2/4  Running automated tests ...


Test ID,Scenario,Input,Expected,Actual,Result
1,Valid IPv4,198.51.100.25,"valid, IPV4",IPV4,PASS
2,Invalid IPv4,999.1.1.1,invalid,rejected,PASS
3,Valid IPv6,2001:db8::1,"valid, IPV6",IPV6,PASS
4,Valid domain,login-check.invalid,"valid, DOMAIN",DOMAIN,PASS
5,Invalid domain,bad_domain..com,invalid,rejected,PASS
6,Valid URL,https://login-check.invalid/verify,"valid, URL",URL,PASS
7,MD5-format hash,32 hex chars,MD5,MD5,PASS
8,SHA1-format hash,40 hex chars,SHA1,SHA1,PASS
9,SHA256-format hash,64 hex chars,SHA256,SHA256,PASS
10,Valid CVE format,CVE-2099-1001,CVE_ID,CVE-2099-1001,PASS


          36/36 tests passed
Step 3/4  Starting the dashboard server ...
Step 4/4  Creating shareable links ...


<IPython.core.display.Javascript object>


Keep this Colab runtime running to keep the dashboard online. Re-run the cell to rebuild everything.
Reminder: all data is synthetic. Indicators are analysed as data and never contacted.
